# DRM · Lab 04 — Assignment 03 Sheet (GRADED)
## Option Pricing and Synthetic Portfolios on **your allotted Nifty 50 stock**

**Course:** DRM (<ECON F354>) · <Semester 1 - 2026-27>
**Instructor:** <Prof. Aditya Sharma> · BITS Pilani
**Total marks:** 45 **Deadline:** `<DD Mon 2026, Midnight>`

---

### What this assignment is

You have to choose one company from the **Nifty 50**. On that stock you will

- estimate volatility from two years of daily prices (**data collection**),
- build a **Black-Scholes-Merton** pricer and a **binomial tree** pricer for European options and
  show that the second converges to the first,
- engineer a **synthetic long call** from the stock and a put using put-call parity, and compare
  it with the actual call — today, over the last three months, and trade-by-trade over two years.

### Before you start

1. Fill in the identification cell below, **including the ticker selected by you**.
   **A submission without it will not be graded.**
2. Write your code in the cells marked `# ---- YOUR CODE HERE ----` and your written answers in
   the markdown cells marked `**Your answer:**`. Do not delete the question cells.
3. **Marks are for interpretation as much as for code.** A correct plot with no commentary earns
   roughly half the marks for that part. Every question that asks "why" or "comment" is looking
   for economics, not for a restatement of what the chart shows.
4. The rubric for each question is printed **directly under that question**. Read it before you
   answer — it tells you exactly what earns the marks.

### Submission

- `Kernel → Restart & Run All`, confirm there are **no errors and all outputs are visible**.
- Save as **`DRM_Lab04_<CampusID>.ipynb`** — e.g. `DRM_Lab04_2023A7PS0123P.ipynb`.
- Upload to the Lab 04 assignment on **Nalanda**. Do not email it; do not open a pull request.

### Marks

| Q | Topic | Marks |
|---|---|---|
| Q1 | Data collection, log returns and annualised volatility | 2 |
| Q2 | Black-Scholes-Merton pricer and Greeks | 5 |
| Q3 | Binomial tree pricer and convergence to BSM | 5 |
| Q4 | Synthetic long call from put-call parity | 5 |
| Q5 | Historical arbitrage simulation: synthetic vs actual call | 8 |
| Q6 | Short answers | 5 |
| | **Total** | **30** |

---
## Evaluation rubric

Each question below carries its own **criterion table** showing how its marks are split. Every
criterion in those tables is graded on the same four-level scale:

| Level | Share of the criterion's marks | Code and output | Written interpretation |
|---|---|---|---|
| **Complete** | 100% | Runs on `Restart & Run All`; result is correct; the required check (assertion, benchmark, table) is present and passes; figure has a title, axis labels with units and a legend | Explains the **mechanism** in economic terms, refers to the student's **own numbers**, and states data provenance (live or simulated) |
| **Substantial** | about 70% | Correct method with one minor slip (a unit, a day-count, an unlabelled axis, a missing check) | Right conclusion, but the mechanism is thin or the numbers are not quoted |
| **Partial** | about 40% | Runs, but a formula or convention is wrong, or part of the required output is missing | Restates what the chart shows without explaining why; or generic text that could belong to any stock |
| **None** | 0% | Missing, does not run, or hard-coded output | Blank, or copied question text |

**How the levels turn into marks.** A criterion worth 2 marks graded *Substantial* earns about
1.5; graded *Partial*, about 1. Marks are awarded in steps of 0.5.

**Notebook-wide rules** (applied after the question marks are added up):

| Notebook-wide issue | Effect |
|---|---|
| Identification cell or honour statement not completed | Not graded |
| `Restart & Run All` raises an error | Cells after the first error are graded at most at the **Partial** level |
| Outputs not visible in the saved file | The affected part is graded as **None** |
| Simulated (`[SIM ]`) or model-generated numbers presented as market data | −2 marks |
| A required source is missing (each of the look-ups listed in the checklist) | −1 mark each |
| Analysis run on a stock other than the one allotted | Not graded |
| AI or outside help used but not disclosed in the declaration | Referred as an integrity violation |
| Late submission | -5 Marks |

**What "interpretation" means here.** Three tests are applied to every written answer:
(i) does it name a mechanism (financing cost, convexity, discreteness of the tree, overlap of
trades…) rather than describe a picture; (ii) does it quote the number *your* notebook produced;
(iii) is it honest about what is model-generated and what is observed.

In [ ]:
# ==================================================================================
#  IDENTIFICATION — fill this in. Submissions without it will not be graded.
# ==================================================================================
NAME       = ""          # e.g. "Ananya Sharma"
CAMPUS_ID  = ""          # e.g. "2023A7PS0123P"
EMAIL      = ""

# The Nifty 50 company allotted to you. Everything below is computed on THIS stock.
TICKER     = "TCS.NS"          # e.g. "HDFCBANK.NS"  (Yahoo symbol = NSE symbol + ".NS")

# Honour statement. Type "I agree" to confirm that the work below is your own and that any
# outside help, source or AI assistance is disclosed in the final cell of this notebook.
HONOUR     = ""

assert NAME and CAMPUS_ID and TICKER and HONOUR.strip().lower() == "i agree", \
    "Fill in NAME, CAMPUS_ID, TICKER and type 'I agree' in HONOUR before running the rest."
print(f"{NAME} | {CAMPUS_ID} | {TICKER}")

---
## Provided setup — run these two cells as-is

The data layer is given to you so that a network failure never blocks the assignment. It tries the
live API first and falls back to a seeded simulator, exactly as in the worked example. **If your
loaders report `[SIM ]`, say so in your write-up** and interpret your results accordingly — you
will not be penalised for working offline, but you will be penalised for presenting simulated
output as though it were market data.

Two conventions are fixed here so that everybody's answers are comparable:

- **Window.** `stock.loc[WIN_START:]` is the two-year analysis window. About four extra months are
  downloaded before it so that rolling volatility is already defined on the first day of the window.
- **Prices.** Closing prices are **not** adjusted for dividends (`auto_adjust=False`), because an
  option's strike is not adjusted for ordinary dividends either.

In [ ]:
# !pip install -q yfinance pandas_datareader        # uncomment in Colab if needed

import warnings, datetime as dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize

warnings.filterwarnings("ignore")
plt.rcParams.update({
    "figure.figsize": (11, 4.4), "figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold", "legend.frameon": False,
})
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

SEED      = 2026
TODAY     = dt.date.today()
END       = TODAY.isoformat()
WIN_START = (TODAY - dt.timedelta(days=730)).isoformat()          # start of the 2-year analysis window
START     = (TODAY - dt.timedelta(days=730 + 120)).isoformat()    # + ~4 months of warm-up for rolling volatility
DATA_MODE = "auto"          # "auto" | "live" | "offline"
IS_SIM = {}

def ttl(title, key=None):
    return title + ("   —   [SIMULATED DATA]" if IS_SIM.get(key) else "")

print(f"pandas {pd.__version__} | download {START} -> {END} | analysis window from {WIN_START} | DATA_MODE = {DATA_MODE}")

In [ ]:
def fetch_yahoo(ticker, start=START, end=END):
    import yfinance as yf
    df = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=False)
    if df is None or len(df) == 0:
        raise RuntimeError("empty frame")
    if isinstance(df.columns, pd.MultiIndex):
        df = df.droplevel(1, axis=1)
    s = df["Close"].astype(float).dropna()
    s.index = pd.to_datetime(s.index).tz_localize(None)
    return s.rename(ticker)

def fetch_fred(series, start=START, end=END):
    from pandas_datareader import data as pdr
    s = pdr.DataReader(series, "fred", start, end).iloc[:, 0].astype(float).dropna()
    s.index = pd.to_datetime(s.index)
    return s.rename(series)

def sim_gbm_series(name, S0, mu, sigma, seed, start=START, end=END):
    idx = pd.bdate_range(start, end)
    rng = np.random.default_rng(seed)
    dtt = 1 / 252
    incr = (mu - 0.5 * sigma**2) * dtt + sigma * np.sqrt(dtt) * rng.standard_normal(len(idx))
    return pd.Series(S0 * np.exp(np.cumsum(incr)), index=idx, name=name)

def sim_ou_series(name, x0, level, kappa, vol, seed, start=START, end=END):
    idx = pd.bdate_range(start, end)
    rng = np.random.default_rng(seed)
    dtt, x, out = 1 / 252, x0, np.empty(len(idx))
    for i in range(len(idx)):
        x += kappa * (level - x) * dtt + vol * np.sqrt(dtt) * rng.standard_normal()
        out[i] = x
    return pd.Series(out, index=idx, name=name)

SIM_ANCHOR = "2021-01-01"      # simulated paths start here, so they do not change from one day to the next

def sim_window(s, S0=None):
    """Cut a simulated path to the download window; optionally rescale it to start at S0."""
    s = s.loc[START:]
    return s if S0 is None else (s / s.iloc[0] * S0).rename(s.name)

def load(key, live_fn, sim_fn):
    if DATA_MODE != "offline":
        try:
            s = live_fn(); IS_SIM[key] = False
            print(f"  [LIVE] {key:<10s} {len(s):>5d} obs  {s.index.min().date()} -> {s.index.max().date()}")
            return s
        except Exception as e:
            if DATA_MODE == "live":
                raise
            print(f"  [SIM ] {key:<10s} live fetch failed ({type(e).__name__}) -> simulated")
    else:
        print(f"  [SIM ] {key:<10s} DATA_MODE=offline")
    IS_SIM[key] = True
    return sim_fn()

print("Loading\n" + "-" * 70)
stock = load("stock", lambda: fetch_yahoo(TICKER),
                      lambda: sim_window(sim_gbm_series(TICKER, 1.0, 0.12, 0.240, SEED + 137, start=SIM_ANCHOR), 1_450.0))
inr_r = load("inr_r", lambda: fetch_fred("INDIRLTLT01STM") / 100.0,
                      lambda: sim_window(sim_ou_series("INR rate", 0.066, 0.066, 1.0, 0.007, SEED + 74, start=SIM_ANCHOR)))
print("-" * 70)

inr_r = inr_r.reindex(stock.index).ffill().bfill()
print(f"{TICKER} {stock.iloc[-1]:,.2f} | r_INR {inr_r.iloc[-1]:.3%} | "
      f"{len(stock.loc[WIN_START:])} trading days in the 2-year window")

---
## Data checks before Q1

Provenance of each series, the TCS dividend calendar (NSE) and the 8% screen on unadjusted closes. Option premiums are model-generated except the Q4(b) quote, which is observed.

In [ ]:
# ==================================================================================
#  ADDED CELL 1: provenance labels, dividend register, corporate-action screen
#  (the provided data-loading cells above are unchanged)
# ==================================================================================
TICKER_SHORT = TICKER.replace(".NS", "")
PROV = {k: ("[SIM]" if v else "[LIVE]") for k, v in IS_SIM.items()}
print("Series provenance:", PROV)
print("Option premiums: model-generated everywhere, except the Q4(b) quote, which is observed (NSE option chain).")
if any(IS_SIM.values()):
    print("\n" + "!" * 100 + "\nWARNING: at least one series is SIMULATED. Every number below is then NOT TCS market data. "
          "Fix the connection and re-run before submitting.\n" + "!" * 100)

def ptitle(title, key="stock", premiums=False):
    """Title with the [LIVE]/[SIM] tag."""
    s = f"{ttl(title, key)}  {PROV[key]}"
    return s + "  | premiums: model-generated" if premiums else s

def expiry_map(idx, horizon_days=30):
    """Expiry = last trading day on or before entry + horizon_days calendar days."""
    target = idx + pd.Timedelta(days=horizon_days)
    pos = idx.searchsorted(target, side="right") - 1
    valid = np.asarray(target <= idx[-1])
    return valid, pos

def fwd_rv(r_full, pos):
    """Annualised realised vol over (entry, expiry]."""
    r = r_full.fillna(0.0).to_numpy()
    c1 = np.concatenate([[0.0], np.cumsum(r)])
    c2 = np.concatenate([[0.0], np.cumsum(r ** 2)])
    i = np.arange(len(r)); n = pos - i
    s1 = c1[pos + 1] - c1[i + 1]
    s2 = c2[pos + 1] - c2[i + 1]
    with np.errstate(divide="ignore", invalid="ignore"):
        var = (s2 - s1 ** 2 / n) / (n - 1)
    return np.where(n > 1, np.sqrt(np.maximum(var, 0.0) * 252), np.nan)

DIV_TABLE = pd.DataFrame([
    ("2023-10-19",  9.0, "interim"), ("2024-01-19",  9.0, "interim"), ("2024-01-19", 18.0, "special"),
    ("2024-05-16", 28.0, "final"),   ("2024-07-19", 10.0, "interim"), ("2024-10-18", 10.0, "interim"),
    ("2025-01-17", 10.0, "interim"), ("2025-01-17", 66.0, "special"), ("2025-06-04", 30.0, "final"),
    ("2025-07-16", 11.0, "interim"), ("2025-10-15", 11.0, "interim"), ("2026-01-16", 11.0, "interim"),
    ("2026-01-16", 46.0, "special"), ("2026-05-25", 31.0, "final"),   ("2026-07-15", 12.0, "interim"),
    ("2026-10-14", 12.0, "interim"),
], columns=["ex_date", "amount", "kind"])
DIV_TABLE["ex_date"] = pd.to_datetime(DIV_TABLE["ex_date"])
DIV_SOURCE = ("NSE corporate actions for TCS, equity segment (CSV downloaded from nseindia.com on 24-Nov-2026); "
              "ex-dates and amounts as listed by NSE")
if stock.index[-1] > DIV_TABLE["ex_date"].max() + pd.Timedelta(days=100):
    print("WARNING: the last ex-date in DIV_TABLE is more than 100 days before the last price date. "
          "Add any newer TCS dividend before relying on q.")

def q_contract(S, tau, r, asof, expiry, div=DIV_TABLE):
    """Continuous yield equivalent to the dividends with ex-date in (asof, expiry]."""
    m = (div["ex_date"] > asof) & (div["ex_date"] <= expiry)
    pv = float((div.loc[m, "amount"] * np.exp(-r * (div.loc[m, "ex_date"] - asof).dt.days / 365)).sum())
    return float(-np.log(1 - pv / S) / tau) if pv > 0 else 0.0

CORP_ACTION_REGISTER = pd.DataFrame([
    ("Buyback (tender offer)", "ex 2023-11-24",  "outside window", "NSE corporate actions, TCS"),
    ("Bonus issue 1:1",        "ex 2018-05-31",  "outside window", "Trendlyne TCS bonus history (read 2026-11-24)"),
    ("Stock split",            "none since 2000","none",           "Trendlyne TCS corporate actions (read 2026-11-24)"),
    ("Bonus / split / demerger / merger / buyback in window", "none", "none",
     "NSE corporate actions list, Nov 2024 to Nov 2026: dividends only"),
    ("Special dividend Rs 66", "ex 2025-01-17",  "in window",      "cash dividend: price drops by the amount on ex-date; not a price-series break"),
    ("Special dividend Rs 46", "ex 2026-01-16",  "in window",      "cash dividend: price drops by the amount on ex-date; not a price-series break"),
], columns=["event", "date", "relation to window", "source / effect"])

FLAG_THRESHOLD = 0.08
STOP_KINDS = {"split", "bonus", "demerger", "merger"}
# flagged days, checked against NSE announcements and news
FLAG_LOG = {
    "2026-06-03": ("market", "IT-sector sell-off on AI-disruption worries and profit booking (Reuters, Business Today, 3-Jun-2026); "
                             "no corporate action, no ex-date"),
}

def screen_corporate_actions(series, threshold=FLAG_THRESHOLD, log=None):
    """Flag |log return| > threshold; stop on unexplained days or price-series breaks."""
    log = FLAG_LOG if log is None else log
    lr = np.log(series).diff().dropna()
    flagged = lr[lr.abs() > threshold]
    rows = []
    for d, v in flagged.items():
        kind, why = log.get(d.strftime("%Y-%m-%d"), ("UNEXPLAINED", ""))
        rows.append({"date": d.date(), "prev_close": series.shift(1).loc[d], "close": series.loc[d],
                     "log_return": v, "kind": kind, "explanation": why})
    out = pd.DataFrame(rows)
    bad = [r for r in rows if r["kind"] in STOP_KINDS or r["kind"] == "UNEXPLAINED"]
    if bad:
        raise RuntimeError(f"STOP: {len(bad)} flagged day(s) are unexplained or are a split/bonus/demerger/merger. "
                           f"Explain each day in FLAG_LOG, or stop if it is a corporate action that breaks the price series.\n{out.to_string()}")
    return out

screen_in_window = stock.loc[WIN_START:]
flag_table = screen_corporate_actions(stock.loc[:])
print(f"\nRule 1 screen on the unadjusted closes {PROV['stock']}: {len(flag_table)} day(s) with |log return| > {FLAG_THRESHOLD:.0%} "
      f"out of {len(stock) - 1} returns (largest absolute daily log return: {np.log(stock).diff().abs().max():.2%}).")
if len(flag_table):
    display(flag_table)

ex_rows = []
for ex, grp in DIV_TABLE.groupby("ex_date"):
    pos = stock.index.searchsorted(ex)
    if ex < pd.Timestamp(WIN_START) or pos == 0 or pos >= len(stock):
        continue
    prev, cur = stock.iloc[pos - 1], stock.iloc[pos]
    ex_rows.append({"ex_date": ex.date(), "cash_dividend_INR": grp["amount"].sum(),
                    "kinds": "+".join(grp["kind"]), "prev_close": prev, "close_on_ex_date": cur,
                    "dividend_pct_of_prev_close": grp["amount"].sum() / prev,
                    "log_return_on_ex_date": np.log(cur / prev)})
ex_table = pd.DataFrame(ex_rows, columns=["ex_date", "cash_dividend_INR", "kinds", "prev_close", "close_on_ex_date",
                                          "dividend_pct_of_prev_close", "log_return_on_ex_date"])
print(f"\nEx-dividend dates inside the analysis window {PROV['stock']} (dividend as % of the previous close; "
      "a dividend is far below the 8% flag, so it does not break the price series):")
display(ex_table)
print("\nCorporate-action register (checked 2026-11-24):")
display(CORP_ACTION_REGISTER)


---
## Q1 · Data collection, log returns and annualised volatility · **[2 marks]**

**(a)** Compute daily **log returns** for your stock. Produce a 2×2 figure over the two-year
window showing: the closing price; the daily log returns; **rolling annualised volatility** on a
21-day and a 63-day window with the full-sample value drawn as a horizontal line; and a histogram
of daily log returns with a normal density of the same mean and standard deviation overlaid.
Label axes with units.

**(b)** Build a summary table containing: number of observations, start and end dates, total
return, **annualised volatility** $\hat\sigma = \text{sd}(\text{log returns})\times\sqrt{252}$,
annualised mean log return, skewness, excess kurtosis and **maximum drawdown**. Store the
volatility in a variable called `SIGMA_HAT` — every later question uses it.

**(c)** In the markdown cell below: report $\hat\sigma$, and the lowest and highest values that the
21-day rolling volatility took in the window. Then answer: **BSM assumes a single constant
$\sigma$. Given your rolling-volatility plot, which estimate would you feed into the pricer for a
one-month option priced today — the two-year figure, the latest 21-day figure, or something else —
and what is the cost of getting it wrong?**

| Part | Rubric — what earns the marks | Marks |
|---|---|---|
| (a) | Log (not simple) returns; all four panels present, correctly annualised, labelled with units; `[SIMULATED DATA]` flag carried in titles if applicable |  |
| (b) | Every statistic present and correct; volatility uses the sample standard deviation and $\sqrt{252}$; drawdown is peak-to-trough and negative; `SIGMA_HAT` defined | 1 |
| (c) | Quotes own $\hat\sigma$ and the rolling range; recognises that volatility is time-varying and clustered; makes a reasoned choice of estimate for a one-month option and links a volatility error to a pricing error through vega | 1 |

In [ ]:
# ---- YOUR CODE HERE ----  Q1 (a) and (b)
# drawdown, skewness, kurtosis and the JB test are adapted from my edhec_risk_kit.py

def drawdown_frame(series):
    """Adapted from edhec_risk_kit.drawdown(): wealth index, running peak and drawdown fraction."""
    wealth = series / series.iloc[0]
    peaks = wealth.cummax()
    return pd.DataFrame({"Wealth": wealth, "Peaks": peaks, "Drawdown": wealth / peaks - 1})

def max_drawdown(series):
    """Return the maximum peak-to-trough decline as a NEGATIVE fraction, e.g. -0.32."""
    return float(drawdown_frame(series)["Drawdown"].min())

def skewness(r):
    """edhec_risk_kit.skewness(): population skewness (ddof=0)."""
    d = r - r.mean()
    return (d ** 3).mean() / r.std(ddof=0) ** 3

def kurtosis(r):
    """edhec_risk_kit.kurtosis(): population kurtosis (a normal distribution gives 3)."""
    d = r - r.mean()
    return (d ** 4).mean() / r.std(ddof=0) ** 4

def jb_pvalue(r):
    """edhec_risk_kit.is_normal() returned p > level; here the p-value itself is returned."""
    return float(stats.jarque_bera(r)[1])

def summary_stats(series):
    """Return a dict with n_obs, start, end, total_return, ann_vol, ann_drift, skew, excess_kurt, max_dd."""
    r = np.log(series).diff().dropna()
    return {
        "n_obs": int(series.shape[0]),
        "start": series.index[0].date(), "end": series.index[-1].date(),
        "total_return": float(series.iloc[-1] / series.iloc[0] - 1),
        "ann_vol": float(r.std(ddof=1) * np.sqrt(252)),
        "ann_drift": float(r.mean() * 252),
        "skew": float(skewness(r)),
        "excess_kurt": float(kurtosis(r) - 3.0),
        "max_dd": max_drawdown(series),
    }

px      = stock.loc[WIN_START:]
ret_all = np.log(stock).diff()
ret     = np.log(px).diff().dropna()
VOL21   = ret_all.rolling(21).std() * np.sqrt(252)
VOL63   = ret_all.rolling(63).std() * np.sqrt(252)

Q1_STATS  = summary_stats(px)
SIGMA_HAT = Q1_STATS["ann_vol"]

assert np.isclose(SIGMA_HAT, np.std(ret.values, ddof=1) * np.sqrt(252))
assert np.isclose(skewness(ret), stats.skew(ret, bias=True))
assert np.isclose(kurtosis(ret) - 3, stats.kurtosis(ret, fisher=True, bias=True))
assert Q1_STATS["max_dd"] <= 0
assert abs(np.log(px).diff().dropna().sum() - np.log(px.iloc[-1] / px.iloc[0])) < 1e-10

r21, r63 = VOL21.loc[WIN_START:] * 100, VOL63.loc[WIN_START:] * 100
fig, ax = plt.subplots(2, 2, figsize=(13, 8))
ax[0, 0].plot(px.index, px.values, color="#1f4e79", lw=1.2, label="Close (unadjusted)")
for ex in ex_table.loc[ex_table["kinds"].str.contains("special"), "ex_date"]:
    ax[0, 0].axvline(pd.Timestamp(ex), color="#b03030", ls="--", lw=1, label="Special-dividend ex-date")
h, l = ax[0, 0].get_legend_handles_labels(); u = dict(zip(l, h))
ax[0, 0].legend(u.values(), u.keys(), loc="best")
ax[0, 0].set_title(ptitle(f"{TICKER_SHORT} closing price")); ax[0, 0].set_xlabel("Date"); ax[0, 0].set_ylabel("Close (INR per share)")

ax[0, 1].plot(ret.index, ret.values * 100, color="#555555", lw=0.7, label="Daily log return")
ax[0, 1].axhline(0, color="black", lw=0.6)
ax[0, 1].set_title(ptitle("Daily log returns")); ax[0, 1].set_xlabel("Date"); ax[0, 1].set_ylabel("Log return (% per day)")
ax[0, 1].legend(loc="best")

ax[1, 0].plot(r21.index, r21.values, color="#d17a00", lw=1.1, label="21-day rolling")
ax[1, 0].plot(r63.index, r63.values, color="#1f4e79", lw=1.4, label="63-day rolling")
ax[1, 0].axhline(SIGMA_HAT * 100, color="#b03030", ls="--", lw=1.2, label=f"Full sample {SIGMA_HAT:.1%}")
ax[1, 0].set_title(ptitle("Rolling annualised volatility")); ax[1, 0].set_xlabel("Date"); ax[1, 0].set_ylabel("Annualised volatility (% p.a.)")
ax[1, 0].legend(loc="best")

xs = np.linspace(ret.min(), ret.max(), 300)
ax[1, 1].hist(ret.values * 100, bins=40, density=True, color="#9db7d1", edgecolor="white", label="Daily log returns")
ax[1, 1].plot(xs * 100, stats.norm.pdf(xs, ret.mean(), ret.std(ddof=1)) / 100, color="#b03030", lw=1.6, label="Normal, same mean and sd")
ax[1, 1].set_title(ptitle("Distribution of daily log returns")); ax[1, 1].set_xlabel("Log return (% per day)"); ax[1, 1].set_ylabel("Density (per % point)")
ax[1, 1].legend(loc="best")
plt.tight_layout(); plt.show()

Q1_TABLE = pd.DataFrame({
    "Statistic": ["Number of daily closes", "Start date", "End date", "Total price return (dividends excluded)",
                  "Annualised volatility (sample sd x sqrt 252)", "Annualised mean log return",
                  "Skewness (population)", "Excess kurtosis (population)", "Maximum drawdown (peak to trough)",
                  "Jarque-Bera p-value (normality)"],
    "Value": [f"{Q1_STATS['n_obs']}", str(Q1_STATS["start"]), str(Q1_STATS["end"]), f"{Q1_STATS['total_return']:.2%}",
              f"{SIGMA_HAT:.2%}", f"{Q1_STATS['ann_drift']:.2%}", f"{Q1_STATS['skew']:.3f}",
              f"{Q1_STATS['excess_kurt']:.3f}", f"{Q1_STATS['max_dd']:.2%}", f"{jb_pvalue(ret):.4g}"],
}).set_index("Statistic")
print(f"Q1(b) summary for {TICKER} {PROV['stock']}")
display(Q1_TABLE)

v21 = VOL21.loc[WIN_START:]
sq = ret ** 2
acf_sq = [float(np.corrcoef(sq.values[:-k], sq.values[k:])[0, 1]) for k in range(1, 6)]
print(f"SIGMA_HAT (2-year)          : {SIGMA_HAT:.2%}")
print(f"21-day rolling vol, lowest  : {v21.min():.2%} on {v21.idxmin().date()}")
print(f"21-day rolling vol, highest : {v21.max():.2%} on {v21.idxmax().date()}")
print(f"21-day rolling vol, latest  : {v21.iloc[-1]:.2%}   | 63-day latest: {VOL63.iloc[-1]:.2%}")
print("Autocorrelation of squared daily returns, lags 1-5 (volatility clustering if > 0): "
      + ", ".join(f"{a:.2f}" for a in acf_sq))


**Your answer (Q1c):**

*Data provenance (live or simulated):*

*$\hat\sigma$ (2-year) = ______ ; 21-day rolling range = ______ to ______*

*Which estimate you would use for a one-month option, and why:*

*Cost of getting $\sigma$ wrong:*

---
## Q2 · Black-Scholes-Merton pricer and Greeks · **[5 marks]**

For a European option on a stock paying a continuous dividend yield $q$:

$$C = S e^{-qT}N(d_1) - K e^{-rT}N(d_2), \qquad P = K e^{-rT}N(-d_2) - S e^{-qT}N(-d_1)$$

$$d_1 = \frac{\ln(S/K) + (r - q + \tfrac12\sigma^2)T}{\sigma\sqrt{T}}, \qquad d_2 = d_1 - \sigma\sqrt{T}$$

**(a)** Implement `bsm_price(S, K, T, r, sigma, q=0, kind="call")` and
`bsm_greeks(S, K, T, r, sigma, q=0, kind="call")` returning delta, gamma, vega and theta. The
function must accept NumPy arrays for `S` and `sigma`. **Include three assertions — numerical
checks, not comments:**
1. the textbook benchmark $S=42,\ K=40,\ r=10\%,\ \sigma=20\%,\ T=0.5$ gives $C=4.76$, $P=0.81$;
2. **put-call parity** $C - P = Se^{-qT} - Ke^{-rT}$ holds to $10^{-8}$ for an arbitrary input with $q \neq 0$;
3. your analytic delta and vega agree with a central **finite difference** of your own price function.

**(b)** Set the **user inputs** for your stock: $S$ = latest close, $K = S$ (at-the-money),
$T = 30/365$, $r$ = latest value of `inr_r`, $\sigma$ = `SIGMA_HAT`, and a dividend yield $q$ that
you **look up and cite**. Report a table with the call premium, the put premium and the four
Greeks of each, with units stated (vega per one volatility point, theta per calendar day).

**(c)** Plot **Delta against stock price** (from 70% to 130% of the strike) for the call and the
put, for three maturities: 1 week, 1 month, 3 months. Plot **Vega against volatility** (from 5% to
80%) for three strikes: 90%, 100% and 110% of spot, at $T = 30/365$.

**(d)** In the markdown cell below, answer all three:
- Why does the delta curve steepen around the strike as expiry approaches, and what does that
  imply for someone hedging a short ATM option in its final week?
- Your ATM call delta is slightly **above** 0.5 even though the option is exactly at the money.
  Show from the formula why.
- Describe the vega-vs-volatility plot: why is the ATM line almost flat while the OTM and ITM lines
  rise from zero? Why do the call and the put with the same strike and expiry have the same vega?

| Part | Rubric — what earns the marks | Marks |
|---|---|---|
| (a) | Correct call and put with dividend yield; vectorised; all three assertions present **and passing** (benchmark, parity, finite difference) | 1 |
| (b) | Inputs taken from own data, not typed in; $q$ cited with a named source; premiums and Greeks tabulated with units | 1 |
| (c) | Delta-vs-$S$ plot (call and put, three maturities) | 1 |
| (c) | Vega-vs-$\sigma$ plot (three strikes), axes in sensible units | 1 |
| (d) | Gamma concentration near expiry and its hedging consequence; $d_1 = (r - q + \sigma^2/2)\sqrt{T}/\sigma > 0$ at the money; vega shape explained and call-put vega equality derived from parity | 1 |

In [ ]:
# ---- YOUR CODE HERE ----  Q2 (a)
from scipy.stats import norm

def _prep(S, K, T, r, sigma, q):
    """Cast inputs to float arrays."""
    S, K, T, r, sigma, q = (np.asarray(x, dtype=float) for x in (S, K, T, r, sigma, q))
    if np.any(sigma <= 0):
        raise ValueError("sigma must be strictly positive")
    return S, K, T, r, sigma, q

def _d1d2(S, K, T, r, sigma, q):
    sq = sigma * np.sqrt(T)
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma ** 2) * T) / sq
    return d1, d1 - sq

def bsm_price(S, K, T, r, sigma, q=0.0, kind="call"):
    """European BSM price with dividend yield q; intrinsic value at T <= 0."""
    S, K, T, r, sigma, q = _prep(S, K, T, r, sigma, q)
    with np.errstate(divide="ignore", invalid="ignore"):
        d1, d2 = _d1d2(S, K, T, r, sigma, q)
        if kind == "call":
            px = S * np.exp(-q * T) * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
            intrinsic = np.maximum(S - K, 0.0)
        elif kind == "put":
            px = K * np.exp(-r * T) * norm.cdf(-d2) - S * np.exp(-q * T) * norm.cdf(-d1)
            intrinsic = np.maximum(K - S, 0.0)
        else:
            raise ValueError("kind must be 'call' or 'put'")
    px = np.where(T <= 0, intrinsic, px)
    return float(px) if px.ndim == 0 else px

def bsm_greeks(S, K, T, r, sigma, q=0.0, kind="call"):
    """Return a dict: delta, gamma, vega (per 1.00 of volatility), theta (per year). Vectorised, T > 0."""
    S, K, T, r, sigma, q = _prep(S, K, T, r, sigma, q)
    d1, d2 = _d1d2(S, K, T, r, sigma, q)
    pdf1, eq, er = norm.pdf(d1), np.exp(-q * T), np.exp(-r * T)
    gamma = eq * pdf1 / (S * sigma * np.sqrt(T))
    vega = S * eq * pdf1 * np.sqrt(T)
    common = -S * eq * pdf1 * sigma / (2 * np.sqrt(T))
    if kind == "call":
        delta = eq * norm.cdf(d1)
        theta = common - r * K * er * norm.cdf(d2) + q * S * eq * norm.cdf(d1)
    elif kind == "put":
        delta = -eq * norm.cdf(-d1)
        theta = common + r * K * er * norm.cdf(-d2) - q * S * eq * norm.cdf(-d1)
    else:
        raise ValueError("kind must be 'call' or 'put'")
    out = {"delta": delta, "gamma": gamma, "vega": vega, "theta": theta}
    return {k: (float(v) if np.ndim(v) == 0 else v) for k, v in out.items()}

_c = bsm_price(42, 40, 0.5, 0.10, 0.20, 0.0, "call"); _p = bsm_price(42, 40, 0.5, 0.10, 0.20, 0.0, "put")
assert round(_c, 2) == 4.76 and round(_p, 2) == 0.81, (_c, _p)

_S = np.array([55.0, 100.0, 2450.0, 17.5]); _K = np.array([60.0, 95.0, 2500.0, 17.5])
_T = np.array([0.25, 1.0, 30 / 365, 2.0]); _r = np.array([0.04, 0.07, 0.065, 0.02])
_sg = np.array([0.35, 0.18, 0.24, 0.50]); _q = np.array([0.03, 0.01, 0.045, 0.06])
_gap = bsm_price(_S, _K, _T, _r, _sg, _q, "call") - bsm_price(_S, _K, _T, _r, _sg, _q, "put") \
       - (_S * np.exp(-_q * _T) - _K * np.exp(-_r * _T))
assert np.max(np.abs(_gap)) < 1e-8, _gap

for _kind in ("call", "put"):
    _g = bsm_greeks(_S, _K, _T, _r, _sg, _q, _kind)
    _h = 1e-4 * _S;  _ds = 1e-5;  _dT = 1e-6
    _up = bsm_price(_S + _h, _K, _T, _r, _sg, _q, _kind); _dn = bsm_price(_S - _h, _K, _T, _r, _sg, _q, _kind)
    _mid = bsm_price(_S, _K, _T, _r, _sg, _q, _kind)
    _delta_fd = (_up - _dn) / (2 * _h)
    _gamma_fd = (_up - 2 * _mid + _dn) / _h ** 2
    _vega_fd = (bsm_price(_S, _K, _T, _r, _sg + _ds, _q, _kind) - bsm_price(_S, _K, _T, _r, _sg - _ds, _q, _kind)) / (2 * _ds)
    _theta_fd = -(bsm_price(_S, _K, _T + _dT, _r, _sg, _q, _kind) - bsm_price(_S, _K, _T - _dT, _r, _sg, _q, _kind)) / (2 * _dT)
    assert np.allclose(_g["delta"], _delta_fd, atol=1e-6), (_kind, _g["delta"], _delta_fd)
    assert np.allclose(_g["vega"], _vega_fd, rtol=1e-6, atol=1e-6), (_kind, _g["vega"], _vega_fd)
    assert np.allclose(_g["gamma"], _gamma_fd, rtol=1e-3, atol=1e-6), (_kind, _g["gamma"], _gamma_fd)
    assert np.allclose(_g["theta"], _theta_fd, rtol=1e-5, atol=1e-5), (_kind, _g["theta"], _theta_fd)
print(f"Q2(a) assertions passed: benchmark C={_c:.4f}, P={_p:.4f}; parity max gap {np.max(np.abs(_gap)):.2e}; "
      "delta, vega (and extra: gamma, theta) match central finite differences.")


In [ ]:
# ---- YOUR CODE HERE ----  Q2 (b) and (c)
DIV_TREATMENT = "TTM_TOTAL"   # trailing-365-day dividends incl. specials / spot; see the dividend-treatment study

S0 = float(stock.iloc[-1])
K0 = S0
T0 = 30 / 365
R0 = float(inr_r.iloc[-1])
SIG0 = SIGMA_HAT

def ttm_dividend_yield(spot_value, asof, kinds=None, div=DIV_TABLE):
    """Cash dividends with ex-date in (asof - 365 days, asof], divided by the spot price. Simple yield, used as q."""
    m = (div["ex_date"] > asof - pd.Timedelta(days=365)) & (div["ex_date"] <= asof)
    if kinds is not None:
        m &= div["kind"].isin(kinds)
    return float(div.loc[m, "amount"].sum() / spot_value)

Q_TOTAL   = ttm_dividend_yield(S0, stock.index[-1])
Q_REGULAR = ttm_dividend_yield(S0, stock.index[-1], kinds=["interim", "final"])
Q_DIV = {"TTM_TOTAL": Q_TOTAL, "TTM_REGULAR": Q_REGULAR, "ZERO": 0.0}[DIV_TREATMENT]
Q_DIV_SOURCE = f"{DIV_SOURCE}. Treatment: {DIV_TREATMENT}; trailing 365 days to {stock.index[-1].date()} divided by the last close."
print(f"q used = {Q_DIV:.4%}  (treatment {DIV_TREATMENT}; total {Q_TOTAL:.4%}, regular only {Q_REGULAR:.4%})")
print("Source:", Q_DIV_SOURCE)

inputs = pd.DataFrame({
    "Input": ["S (latest close, INR)", "K = S (INR)", "T (years)", "r (latest inr_r)", "sigma (SIGMA_HAT)", "q (dividend yield)"],
    "Value": [f"{S0:,.2f}", f"{K0:,.2f}", f"{T0:.5f}", f"{R0:.3%}", f"{SIG0:.2%}", f"{Q_DIV:.3%}"],
    "Provenance": [PROV["stock"], PROV["stock"], "30/365", PROV["inr_r"], PROV["stock"] + " (Q1)", "cited source above"],
}).set_index("Input")
display(inputs)

rows = {}
for kind in ("call", "put"):
    g = bsm_greeks(S0, K0, T0, R0, SIG0, Q_DIV, kind)
    rows[kind] = {"Premium (INR, model-generated)": bsm_price(S0, K0, T0, R0, SIG0, Q_DIV, kind),
                  "Delta (per INR move in S)": g["delta"],
                  "Gamma (delta change per INR)": g["gamma"],
                  "Vega (INR per 1 vol point)": g["vega"] / 100,
                  "Theta (INR per calendar day)": g["theta"] / 365}
Q2_TABLE = pd.DataFrame(rows).T
Q2_TABLE.index = ["ATM call", "ATM put"]
print(f"Q2(b): ATM options on {TICKER} {PROV['stock']} underlying, model-generated premiums")
display(Q2_TABLE)
C_ATM, P_ATM = Q2_TABLE.iloc[0, 0], Q2_TABLE.iloc[1, 0]
assert abs((C_ATM - P_ATM) - (S0 * np.exp(-Q_DIV * T0) - K0 * np.exp(-R0 * T0))) < 1e-8

Ss = np.linspace(0.7 * K0, 1.3 * K0, 241)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6), sharey=False)
for (lab, Tm), col in zip([("1 week", 7 / 365), ("1 month", 30 / 365), ("3 months", 91 / 365)], ["#d17a00", "#1f4e79", "#2e7d32"]):
    ax[0].plot(Ss, bsm_greeks(Ss, K0, Tm, R0, SIG0, Q_DIV, "call")["delta"], color=col, lw=1.6, label=f"T = {lab}")
    ax[1].plot(Ss, bsm_greeks(Ss, K0, Tm, R0, SIG0, Q_DIV, "put")["delta"], color=col, lw=1.6, label=f"T = {lab}")
for a, nm in zip(ax, ["Call", "Put"]):
    a.axvline(K0, color="grey", ls=":", lw=1, label="Strike K")
    a.set_title(ptitle(f"{nm} delta against stock price", premiums=True)); a.set_xlabel("Stock price S (INR per share)")
    a.set_ylabel("Delta (change in option value per INR)"); a.legend(loc="best")
plt.tight_layout(); plt.show()

sg = np.linspace(0.05, 0.80, 200)
fig, ax = plt.subplots(figsize=(8.5, 4.6))
for m, col in zip([0.90, 1.00, 1.10], ["#d17a00", "#1f4e79", "#2e7d32"]):
    ax.plot(sg * 100, bsm_greeks(S0, m * S0, 30 / 365, R0, sg, Q_DIV, "call")["vega"] / 100, color=col, lw=1.6,
            label=f"K = {m:.0%} of spot (call and put identical)")
ax.set_title(ptitle("Vega against volatility, T = 30/365", premiums=True))
ax.set_xlabel("Volatility sigma (% per annum)"); ax.set_ylabel("Vega (INR per 1 volatility point)"); ax.legend(loc="best")
plt.tight_layout(); plt.show()

d1_atm = (R0 - Q_DIV + 0.5 * SIG0 ** 2) * np.sqrt(T0) / SIG0
print(f"d1 at the money = (r - q + sigma^2/2) sqrt(T) / sigma = {d1_atm:.4f}  ->  N(d1) = {norm.cdf(d1_atm):.4f}; "
      f"call delta = e^(-qT) N(d1) = {Q2_TABLE.iloc[0, 1]:.4f}")
print(f"Put-call vega check (identical by parity): call {bsm_greeks(S0, K0, T0, R0, SIG0, Q_DIV, 'call')['vega']:.4f}, "
      f"put {bsm_greeks(S0, K0, T0, R0, SIG0, Q_DIV, 'put')['vega']:.4f}")


**Your answer (Q2):**

*Dividend yield used and source:*

*(d-i) Why delta steepens near expiry, and what it means for a hedger:*

*(d-ii) Why the ATM call delta exceeds 0.5:*

*(d-iii) Shape of vega vs volatility, and why call vega = put vega:*

---
## Supplement B · Treatment of TCS's special dividends in $q$

TCS paid special dividends on 17-Jan-2025 and 16-Jan-2026. This compares ways of setting the single yield $q$ against the dated cash dividends. The choice in `DIV_TREATMENT` is used in Q2, Q4 and Q5.

In [ ]:
# ==================================================================================
#  SUPPLEMENT B: how should q treat the special dividends?  (my addition; the choice is then fixed for Q2, Q4 and Q5)
# ==================================================================================
#   credit error   = S0 (e^(qT) - 1) - actual cash dividends received          (rule 4 credit against the real cash)
#   spurious gap   = S e^(-qT) - S*   = the put-call parity gap that a wrong q creates by itself, in bp of spot
#   price error    = BSM(q) ATM call - benchmark ATM call
TC_BPS = 25.0

def dividend_study(spot, rate, vol, div=DIV_TABLE, horizon_days=30):
    idx = spot.index; valid, pos = expiry_map(idx, horizon_days)
    keep = valid & (idx >= pd.Timestamp(WIN_START)) & vol.notna().to_numpy()
    ei = np.where(keep)[0]; xi = pos[ei]
    S = spot.to_numpy()[ei]; r_ = rate.reindex(idx).to_numpy()[ei]; sg = vol.to_numpy()[ei]
    T = np.array([(idx[b] - idx[a]).days for a, b in zip(ei, xi)]) / 365
    ent = idx.to_numpy()[ei][:, None]; exp_ = idx.to_numpy()[xi][:, None]
    ex = div["ex_date"].to_numpy()[None, :]; amt = div["amount"].to_numpy()[None, :]
    spec = (div["kind"].to_numpy()[None, :] == "special")
    inwin = (ex > ent) & (ex <= exp_)
    yrs = ((ex - ent) / np.timedelta64(1, "D")) / 365
    D_true = (inwin * amt).sum(1); PVD = (inwin * amt * np.exp(-r_[:, None] * yrs)).sum(1)
    S_star = S - PVD
    C_d = bsm_price(S_star, S, T, r_, sg, 0.0, "call")
    grp = np.where((inwin & spec).any(1), "ex-date inside, special", np.where(inwin.any(1), "ex-date inside, regular only", "no ex-date inside"))
    back = (ex <= ent) & (ex > ent - np.timedelta64(365, "D"))
    q_roll = (back * amt).sum(1) / S
    qs = {"q = 0": np.zeros_like(S), "TTM regular only (constant)": np.full_like(S, Q_REGULAR),
          "TTM incl. special (constant)": np.full_like(S, Q_TOTAL), "Rolling TTM incl. special (causal)": q_roll}
    rows, by_grp = [], []
    for nm, q in qs.items():
        credit = S * (np.exp(q * T) - 1); cerr = credit - D_true
        gap = S * np.exp(-q * T) - S_star; gap_bp = gap / S * 1e4
        C_q = bsm_price(S, S, T, r_, sg, q, "call"); perr = (C_q - C_d) / C_d
        rows.append({"Treatment of q": nm, "Mean q": q.mean(), "Credit error, mean (INR)": cerr.mean(), "Credit error, RMSE (INR)": np.sqrt((cerr ** 2).mean()),
                     "Spurious parity gap, mean (bp)": gap_bp.mean(), "Spurious gap, RMSE (bp)": np.sqrt((gap_bp ** 2).mean()),
                     "Share |gap| > 25 bp": (np.abs(gap_bp) > TC_BPS).mean(),
                     "ATM call price error, mean (%)": perr.mean() * 100, "ATM call price error, RMSE (%)": np.sqrt((perr ** 2).mean()) * 100})
        for g in np.unique(grp):
            m = grp == g
            by_grp.append({"Treatment of q": nm, "Trades": g, "n": int(m.sum()), "Credit error, mean (INR)": cerr[m].mean(),
                           "Spurious gap, mean (bp)": gap_bp[m].mean(), "Spurious gap, RMSE (bp)": np.sqrt((gap_bp[m] ** 2).mean())})
    return pd.DataFrame(rows).set_index("Treatment of q"), pd.DataFrame(by_grp).set_index(["Treatment of q", "Trades"]), pd.Series(grp).value_counts()

DIV_STUDY, DIV_STUDY_GRP, grp_counts = dividend_study(stock, inr_r, VOL21)
print(f"Supplement B on {TICKER} {PROV['stock']} with the dated cash-dividend calendar; every option price here is model-generated")
print("Trades by dividend exposure:", grp_counts.to_dict())
print("\n(1) Whole window. Smaller is better in every column except Mean q. If the stock leg were credited with the REAL cash dividend while "
      "the premiums kept a smooth q, financed-minus-call would equal this spurious gap, so the rule-4 credit S0(e^(qT)-1) must use the same q as the premiums.")
display(DIV_STUDY)
print("(2) By exposure group:")
display(DIV_STUDY_GRP)

q_today_ctr = q_contract(S0, T0, R0, stock.index[-1], stock.index[-1] + pd.Timedelta(days=30))
today = {}
for nm, qq in (("q = 0", 0.0), ("TTM regular only", Q_REGULAR), ("TTM incl. special", Q_TOTAL), ("contract-specific (ex-dates before expiry)", q_today_ctr)):
    c, p = bsm_price(S0, K0, T0, R0, SIG0, qq, "call"), bsm_price(S0, K0, T0, R0, SIG0, qq, "put")
    today[nm] = {"q": qq, "ATM call (INR)": c, "ATM put (INR)": p, "Parity RHS S e^(-qT) - K e^(-rT) (INR)": S0 * np.exp(-qq * T0) - K0 * np.exp(-R0 * T0)}
TODAY_TABLE = pd.DataFrame(today).T
TODAY_TABLE["Call vs contract-specific (%)"] = (TODAY_TABLE["ATM call (INR)"] / TODAY_TABLE.loc["contract-specific (ex-dates before expiry)", "ATM call (INR)"] - 1) * 100
print(f"\n(3) The option priced today on {stock.index[-1].date()} (30 days to {(stock.index[-1] + pd.Timedelta(days=30)).date()}); "
      f"dividends with an ex-date before that expiry in DIV_TABLE: "
      f"{int(((DIV_TABLE['ex_date'] > stock.index[-1]) & (DIV_TABLE['ex_date'] <= stock.index[-1] + pd.Timedelta(days=30))).sum())}")
display(TODAY_TABLE)
print("If a dividend is announced inside the next 30 days, add it to DIV_TABLE before running: contract-specific q then becomes positive.")


---
## Q3 · Binomial tree pricer and convergence to BSM · **[5 marks]**

Use the Cox-Ross-Rubinstein parameterisation with $N$ steps of length $\Delta t = T/N$:

$$u = e^{\sigma\sqrt{\Delta t}}, \qquad d = 1/u, \qquad p = \frac{e^{(r-q)\Delta t} - d}{u - d}$$

**(a)** Implement `binomial_price(S, K, T, r, sigma, N, q=0, kind="call")` for **European**
options by backward induction. The function must raise an error if the risk-neutral probability
is not strictly between 0 and 1. Do not use the closed-form binomial sum, and do not loop over
nodes one at a time in Python — work on a whole time-slice as an array.

**(b)** With the same user inputs as Q2(b), price the ATM call and put with **$N = 50$** and
$N = 500$. Report a table of binomial price, BSM price, and the difference in rupees and in
basis points of the BSM price. **Assert** that your tree satisfies put-call parity to $10^{-8}$
at $N=50$ — it must, for any $N$.

**(c)** Convergence. Compute the call price for every $N$ from 1 to 400 and plot (i) the binomial
price against $N$ with the BSM value as a horizontal line, and (ii) the absolute error against $N$
on log-log axes with a reference line of slope $-1$. Repeat (i) for a strike **5% out of the
money**. Add a table for $N \in \{10, 25, 50, 51, 100, 101, 200, 201, 400\}$.

**(d)** In the markdown cell below, answer all three:
- At what rate does the error fall with $N$? Read it off your log-log plot, and say how many steps
  you need for the price to be within ₹0.01 of BSM.
- The ATM error **zig-zags** between odd and even $N$, and the OTM error moves in **waves**. What
  is the cause? Name one simple fix and show numerically that it works.
- Parity holds exactly in your tree even at $N=5$, where the price itself is poor. Why?

| Part | Rubric — what earns the marks | Marks |
|---|---|---|
| (a) | Correct CRR backward induction, vectorised across a time-slice, dividend yield in $p$, validity check on $p$; $N \ge 50$ supported | 1 |
| (b) | Table at $N=50$ and $N=500$ against BSM; parity assertion on the tree present and passing | 1 |
| (c) | Both convergence plots, the OTM comparison and the table; log-log axes and reference slope | 1 |
| (d) | $O(1/N)$ rate read from own plot and translated into a step count; oscillation attributed to the position of the strike relative to the terminal nodes, with a fix demonstrated; exact parity explained by linearity of risk-neutral expectation | 2 |

In [ ]:
# ---- YOUR CODE HERE ----  Q3 (a) and (b)

def binomial_price(S, K, T, r, sigma, N=50, q=0.0, kind="call"):
    """CRR tree, European exercise, backward induction one whole time-slice at a time (no loop over nodes)."""
    if N < 1 or int(N) != N:
        raise ValueError("N must be a positive integer")
    N = int(N)
    dt = T / N
    u = np.exp(sigma * np.sqrt(dt)); d = 1.0 / u
    p = (np.exp((r - q) * dt) - d) / (u - d)
    if not (0.0 < p < 1.0):
        raise ValueError(f"risk-neutral probability p = {p:.6f} is outside (0, 1): increase N or check r, q, sigma")
    disc = np.exp(-r * dt)
    j = np.arange(N + 1)
    ST = S * np.exp((2 * j - N) * sigma * np.sqrt(dt))
    V = np.maximum(ST - K, 0.0) if kind == "call" else np.maximum(K - ST, 0.0)
    if kind not in ("call", "put"):
        raise ValueError("kind must be 'call' or 'put'")
    for _ in range(N):
        V = disc * (p * V[1:] + (1.0 - p) * V[:-1])
    return float(V[0])

rows = []
for kind in ("call", "put"):
    bs = bsm_price(S0, K0, T0, R0, SIG0, Q_DIV, kind)
    for N in (50, 500):
        tree = binomial_price(S0, K0, T0, R0, SIG0, N, Q_DIV, kind)
        rows.append({"Option": f"ATM {kind}", "N": N, "Binomial (INR)": tree, "BSM (INR)": bs,
                     "Difference (INR)": tree - bs, "Difference (bp of BSM price)": (tree - bs) / bs * 1e4})
Q3_TABLE = pd.DataFrame(rows).set_index(["Option", "N"])
print(f"Q3(b): tree against closed form for {TICKER} {PROV['stock']} underlying, model-generated premiums")
display(Q3_TABLE)

c50 = binomial_price(S0, K0, T0, R0, SIG0, 50, Q_DIV, "call"); p50 = binomial_price(S0, K0, T0, R0, SIG0, 50, Q_DIV, "put")
parity_rhs = S0 * np.exp(-Q_DIV * T0) - K0 * np.exp(-R0 * T0)
assert abs((c50 - p50) - parity_rhs) < 1e-8, (c50 - p50, parity_rhs)
print(f"Tree parity at N=50: C - P = {c50 - p50:.10f}, S e^(-qT) - K e^(-rT) = {parity_rhs:.10f}, gap {abs((c50 - p50) - parity_rhs):.2e}")
try:
    binomial_price(100, 100, 1.0, 0.50, 0.01, 1, 0.0, "call"); raise AssertionError("validity check did not fire")
except ValueError as e:
    print("Validity check on p works:", str(e)[:70], "...")


In [ ]:
# ---- YOUR CODE HERE ----  Q3 (c): convergence plots and table
Ns = np.arange(1, 401)
K_OTM = 1.05 * K0
bs_atm = bsm_price(S0, K0, T0, R0, SIG0, Q_DIV, "call")
bs_otm = bsm_price(S0, K_OTM, T0, R0, SIG0, Q_DIV, "call")
tree_atm = np.array([binomial_price(S0, K0, T0, R0, SIG0, n, Q_DIV, "call") for n in Ns])
tree_otm = np.array([binomial_price(S0, K_OTM, T0, R0, SIG0, n, Q_DIV, "call") for n in Ns])
err_atm, err_otm = tree_atm - bs_atm, tree_otm - bs_otm

c_ref = float(np.median(np.abs(err_atm[19:]) * Ns[19:]))

fig, ax = plt.subplots(2, 2, figsize=(13, 8.5))
ax[0, 0].plot(Ns, tree_atm, color="#1f4e79", lw=0.9, label="Binomial call, K = S")
ax[0, 0].axhline(bs_atm, color="#b03030", ls="--", lw=1.3, label=f"BSM = {bs_atm:,.2f}")
ax[0, 0].set_title(ptitle("ATM call: binomial price against N", premiums=True)); ax[0, 0].set_xlabel("Number of steps N"); ax[0, 0].set_ylabel("Call price (INR per share)")
ax[0, 0].legend(loc="best")
ax[0, 1].plot(Ns, tree_otm, color="#2e7d32", lw=0.9, label="Binomial call, K = 1.05 S")
ax[0, 1].axhline(bs_otm, color="#b03030", ls="--", lw=1.3, label=f"BSM = {bs_otm:,.2f}")
ax[0, 1].set_title(ptitle("5% OTM call: binomial price against N", premiums=True)); ax[0, 1].set_xlabel("Number of steps N"); ax[0, 1].set_ylabel("Call price (INR per share)")
ax[0, 1].legend(loc="best")
ax[1, 0].loglog(Ns, np.abs(err_atm), color="#1f4e79", lw=0.8, label="|error|, ATM")
ax[1, 0].loglog(Ns, np.abs(err_otm), color="#2e7d32", lw=0.8, label="|error|, 5% OTM")
ax[1, 0].loglog(Ns, c_ref / Ns, color="black", ls="--", lw=1.2, label="Reference slope -1 (c/N)")
ax[1, 0].set_title(ptitle("Absolute error against N (log-log)", premiums=True)); ax[1, 0].set_xlabel("Number of steps N (log scale)"); ax[1, 0].set_ylabel("|Binomial - BSM| (INR, log scale)")
ax[1, 0].legend(loc="best")

tree_atm_next = np.array([binomial_price(S0, K0, T0, R0, SIG0, n + 1, Q_DIV, "call") for n in Ns])
tree_otm_next = np.array([binomial_price(S0, K_OTM, T0, R0, SIG0, n + 1, Q_DIV, "call") for n in Ns])
avg_atm, avg_otm = 0.5 * (tree_atm + tree_atm_next), 0.5 * (tree_otm + tree_otm_next)
ax[1, 1].loglog(Ns, np.abs(err_atm), color="#9db7d1", lw=0.8, label="|error| ATM, raw tree")
ax[1, 1].loglog(Ns, np.abs(avg_atm - bs_atm), color="#1f4e79", lw=1.1, label="|error| ATM, average of N and N+1")
ax[1, 1].loglog(Ns, np.abs(err_otm), color="#a5d6a7", lw=0.8, label="|error| OTM, raw tree")
ax[1, 1].loglog(Ns, np.abs(avg_otm - bs_otm), color="#2e7d32", lw=1.1, label="|error| OTM, average of N and N+1")
ax[1, 1].set_title(ptitle("Fix: average the trees with N and N+1 steps", premiums=True)); ax[1, 1].set_xlabel("Number of steps N (log scale)"); ax[1, 1].set_ylabel("|Error| against BSM (INR, log scale)")
ax[1, 1].legend(loc="best", fontsize=8)
plt.tight_layout(); plt.show()

sel = [10, 25, 50, 51, 100, 101, 200, 201, 400]
Q3C_TABLE = pd.DataFrame({
    "ATM binomial (INR)": [tree_atm[n - 1] for n in sel], "ATM error (INR)": [err_atm[n - 1] for n in sel],
    "ATM avg(N, N+1) error (INR)": [avg_atm[n - 1] - bs_atm for n in sel],
    "OTM binomial (INR)": [tree_otm[n - 1] for n in sel], "OTM error (INR)": [err_otm[n - 1] for n in sel],
    "OTM avg(N, N+1) error (INR)": [avg_otm[n - 1] - bs_otm for n in sel]}, index=pd.Index(sel, name="N"))
print(f"BSM: ATM call {bs_atm:,.4f}, 5% OTM call {bs_otm:,.4f} INR (model-generated, underlying {PROV['stock']})")
display(Q3C_TABLE)

conv_slope = np.polyfit(np.log(Ns[19:]), np.log(np.abs(err_atm[19:]) + 1e-300), 1)[0]
print(f"Fitted slope of log|error| on log N, ATM, N >= 20 : {conv_slope:.2f}   (a perfect 1/N rate gives -1; the zig-zag blurs the fit)")
print(f"Error band: c/N with c = {c_ref:.3f} INR.  Steps for |error| <= 0.01 INR: about N = {int(np.ceil(c_ref / 0.01)):,} (extrapolated)")
ok = np.where(np.abs(err_atm) <= 0.01)[0]
print("Smallest N from which the raw ATM error stays within 0.01 INR up to N=400:",
      int(Ns[ok[0]]) if len(ok) and np.all(np.abs(err_atm[ok[0]:]) <= 0.01) else "not reached within N=400")
zig_raw = np.mean(np.abs(np.diff(err_atm[49:])));  zig_avg = np.mean(np.abs(np.diff(avg_atm[49:] - bs_atm)))
print(f"Mean |change in ATM error between N and N+1|, N >= 50: raw tree {zig_raw:.4f} INR vs averaged {zig_avg:.4f} INR "
      f"({zig_avg / zig_raw:.0%} of raw)")
print(f"Mean |ATM error|, N = 50..400: raw {np.mean(np.abs(err_atm[49:])):.4f} INR, averaged {np.mean(np.abs(avg_atm[49:] - bs_atm)):.4f} INR")
print(f"Mean |OTM error|, N = 50..400: raw {np.mean(np.abs(err_otm[49:])):.4f} INR, averaged {np.mean(np.abs(avg_otm[49:] - bs_otm)):.4f} INR")


**Your answer (Q3d):**

*Rate of convergence and the number of steps needed for ₹0.01 accuracy:*

*Cause of the zig-zag (ATM) and the waves (OTM); one fix and the evidence that it works:*

*Why parity is exact on the tree for any $N$:*

---
## Q4 · Synthetic long call from put-call parity · **[5 marks]**

Put-call parity for European options with the same strike and expiry:

$$C - P = S e^{-qT} - K e^{-rT} \quad\Longleftrightarrow\quad C = S e^{-qT} + P - K e^{-rT}$$

The strategy to build is **long stock + long put**, with the put struck at the same level and
expiring on the same day as the call you are replicating.

**(a) Cost and payoff today.** Take the ATM call from Q2(b) ($K = S$, $T = 30/365$). Using your
own pricer, tabulate the **cost today** and the **payoff at expiry** of three portfolios:
(1) the actual call; (2) the synthetic *stock + put*; (3) the *financed* synthetic
*stock + put − loan of $Ke^{-rT}$*. Plot the expiry **payoff** and the expiry **profit** of all
three against $S_T$ from 80% to 120% of the strike.

**(b) An observed quote.** Obtain the **observed** premiums of a near-the-money call and put on
your stock with the same strike and expiry — from the NSE option chain, your broker's terminal or
an API — together with the spot price **at the same moment**. Record strike, expiry date,
timestamp and source. Compute:
- the **parity gap** $C^{obs} - P^{obs} - (S e^{-q\tau} - K e^{-r\tau})$ in rupees and in bps of spot;
- the **implied financing rate** $r_{\text{implied}} = -\frac{1}{\tau}\ln\!\big[(S e^{-q\tau} - C^{obs} + P^{obs})/K\big]$;
- the **implied volatility** of the call and of the put, by root-finding on your `bsm_price`.

Compare the implied volatilities with `SIGMA_HAT`, and the implied financing rate with `inr_r`.

**(c) Track over the last three months.** Take the last three consecutive 30-day cycles ending
on the latest date in your data. In each cycle, open the actual call and the synthetic at the
close of the first day ($K$ = that day's close), **mark both to model every day** to expiry
(using that day's spot, rate, remaining time and 21-day rolling volatility), then roll. Plot the
cumulative P&L per share of the actual call, the synthetic and the financed synthetic. Report a
per-cycle table: entry and expiry dates, $S_0$, $S_T$, $C_0$, $P_0$, and the P&L and return on
capital of each portfolio.

**(d)** In the markdown cell below, **explain the discrepancies**:
- The synthetic *stock + put* costs far more than the call and pays $K$ more at expiry. Which
  portfolio is the true replica of the call, and what does the difference in **profit** between
  portfolios (1) and (2) represent? Give it as a formula and as a rupee figure from your table.
- In (c) the gap between the synthetic and the call is not constant through a cycle. Derive what
  it equals on any day, and confirm it against your series.
- In (b), is your parity gap large enough to trade? Name the arbitrage you would put on
  (conversion or reversal), and give **three** reasons the gap may not be capturable.
- Your call and put implied volatilities differ from `SIGMA_HAT`. Which way, and why is that the
  normal state of affairs?

| Part | Rubric — what earns the marks | Marks |
|---|---|---|
| (a) | Cost and payoff table for all three portfolios; payoff and profit diagrams; shows payoff(2) = payoff(1) + $K$ and cost(3) = cost(1) | 1 |
| (b) | Real quote with strike, expiry, timestamp and source, spot taken at the same moment; parity gap, implied financing rate and both implied volatilities computed by root-finding. *If no live quote could be obtained, a clearly labelled model-generated quote earns at most 1* | 1 |
| (c) | Three non-overlapping cycles; daily mark-to-model with shrinking time to expiry and no look-ahead; cumulative P&L chart and per-cycle table with return on capital | 1 |
| (d) | Identifies the financed portfolio as the replica and the profit gap as the financing cost $S_0 - Ke^{-rT}$ (less dividends); derives the daily gap from parity; realistic view on capturing the observed gap (bid-ask, margin and short-selling constraints, asynchronous quotes, dividends, taxes and fees); implied-vs-realised volatility explained | 2 |

In [ ]:
# ---- YOUR CODE HERE ----  Q4 (a): cost table, payoff and profit diagrams
#   (1) actual call
#   (2) synthetic: buy the stock + buy the put; the stock leg is credited with D = S0 (e^(qT) - 1) of dividends (rule 4)
#   (3) financed synthetic: (2) minus a loan of K e^(-rT), repaid with K at expiry
#   (4) dividend-tailed replica: e^(-qT) shares + put - loan; dividends are reinvested, so it ends with one share.
#       Added because cost(3) differs from cost(1) by the dividend wedge S0 (1 - e^(-qT)) when q != 0.
D0 = S0 * (np.exp(Q_DIV * T0) - 1)
loan = K0 * np.exp(-R0 * T0)
cost  = {"(1) Actual call": C_ATM,
         "(2) Synthetic: stock + put": S0 + P_ATM,
         "(3) Financed synthetic: stock + put - loan": S0 + P_ATM - loan,
         "(4) Tailed replica: e^(-qT) stock + put - loan": S0 * np.exp(-Q_DIV * T0) + P_ATM - loan}

def payoff_at_expiry(ST):
    ST = np.asarray(ST, dtype=float)
    call = np.maximum(ST - K0, 0.0)
    put = np.maximum(K0 - ST, 0.0)
    return {"(1) Actual call": call,
            "(2) Synthetic: stock + put": ST + D0 + put,
            "(3) Financed synthetic: stock + put - loan": ST + D0 + put - K0,
            "(4) Tailed replica: e^(-qT) stock + put - loan": ST + put - K0}

ST_grid = np.linspace(0.8 * K0, 1.2 * K0, 241)
pay = payoff_at_expiry(ST_grid)
prof = {k: pay[k] - cost[k] for k in pay}

assert np.allclose(pay["(2) Synthetic: stock + put"] - pay["(1) Actual call"], K0 + D0)
assert np.allclose(pay["(4) Tailed replica: e^(-qT) stock + put - loan"], pay["(1) Actual call"])
assert abs(cost["(4) Tailed replica: e^(-qT) stock + put - loan"] - cost["(1) Actual call"]) < 1e-8
assert abs(cost["(3) Financed synthetic: stock + put - loan"] - cost["(1) Actual call"] - S0 * (1 - np.exp(-Q_DIV * T0))) < 1e-8
if Q_DIV == 0:
    assert abs(cost["(3) Financed synthetic: stock + put - loan"] - cost["(1) Actual call"]) < 1e-8

pts = {"S_T = 90% K": 0.9 * K0, "S_T = K": K0, "S_T = 110% K": 1.1 * K0}
tab = pd.DataFrame({"Cost today (INR)": cost})
for lab, st in pts.items():
    tab[f"Payoff at {lab} (INR)"] = [payoff_at_expiry(st)[k] for k in cost]
tab["Profit at S_T = K (INR)"] = [payoff_at_expiry(K0)[k] - cost[k] for k in cost]
print(f"Q4(a): per share, K = S0 = {S0:,.2f}, T = 30/365; stock {PROV['stock']}, premiums model-generated; dividend credit D = {D0:,.4f} INR")
display(tab)
print(f"payoff(2) - payoff(1) = K + D = {K0 + D0:,.4f} INR at every S_T (asserted)")
print(f"cost(3) - cost(1) = S0 (1 - e^(-qT)) = {cost['(3) Financed synthetic: stock + put - loan'] - cost['(1) Actual call']:,.4f} INR (dividend wedge); cost(4) = cost(1)")
print(f"Profit gap (2) - (1) at S_T = K: {prof['(2) Synthetic: stock + put'][120] - prof['(1) Actual call'][120]:,.4f} INR   [= K + D - S0 - P0 + C0]")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
sty = {"(1) Actual call": ("#1f4e79", "-"), "(2) Synthetic: stock + put": ("#d17a00", "-"),
       "(3) Financed synthetic: stock + put - loan": ("#2e7d32", "--"), "(4) Tailed replica: e^(-qT) stock + put - loan": ("#b03030", ":")}
for k, (c, ls) in sty.items():
    ax[0].plot(ST_grid, pay[k], color=c, ls=ls, lw=1.6, label=k)
    ax[1].plot(ST_grid, prof[k], color=c, ls=ls, lw=1.6, label=k)
ax[0].set_title(ptitle("Payoff at expiry", premiums=True)); ax[1].set_title(ptitle("Profit at expiry (payoff - cost today)", premiums=True))
ax[0].set_ylabel("Payoff (INR per share)"); ax[1].set_ylabel("Profit (INR per share)")
for a in ax:
    a.set_xlabel("Stock price at expiry S_T (INR per share)"); a.axvline(K0, color="grey", ls=":", lw=1); a.legend(loc="best", fontsize=8)
plt.tight_layout(); plt.show()


In [ ]:
# ---- YOUR CODE HERE ----  Q4 (b): observed quote, parity gap, implied financing rate, implied vols
C_OBS        = 99.75
P_OBS        = 82.00
K_OBS        = 2460.00
S_OBS        = 2460.00
EXPIRY_OBS   = "2026-12-29"
QUOTE_ASOF   = "2026-11-23 15:40 IST"
QUOTE_SOURCE = ("NSE option chain (equity derivatives), nseindia.com/option-chain, TCS, expiry 29-Dec-2026, "
                "CSV and screenshot saved 23/24-Nov-2026")
QUOTE_TYPE   = "LTP (last traded price). Bid and ask were blank in the after-hours download, so no mid-price was available."
NSE_IV_CALL, NSE_IV_PUT = 0.2993, 0.2911
VOL_CALL, VOL_PUT = 5141, 2749
OI_CALL, OI_PUT   = 780, 448

def implied_vol(price, S, K, T, r, q=0.0, kind="call"):
    """Root-find sigma with Brent's method so that bsm_price(...) equals the observed price. NaN if no sigma fits."""
    f = lambda s: bsm_price(S, K, T, r, s, q, kind) - price
    lo, hi = 1e-4, 5.0
    if f(lo) > 0 or f(hi) < 0:
        return np.nan
    return float(optimize.brentq(f, lo, hi, xtol=1e-12))

assert abs(implied_vol(bsm_price(100, 105, 0.25, 0.06, 0.31, 0.02, "put"), 100, 105, 0.25, 0.06, 0.02, "put") - 0.31) < 1e-8

QUOTE_READY = all(v not in (None, "") for v in (C_OBS, P_OBS, K_OBS, S_OBS, EXPIRY_OBS, QUOTE_ASOF, QUOTE_SOURCE))
if not QUOTE_READY:
    print("Q4(b): observed-quote constants are still empty, so this part is skipped. Fill them from the NSE option chain "
          "and re-run; nothing else in the notebook depends on them.")
else:
    asof = pd.Timestamp(QUOTE_ASOF[:10]); expiry = pd.Timestamp(EXPIRY_OBS)
    tau = (expiry - asof).days / 365
    r_q = float(inr_r.iloc[-1])
    q_sheet = Q_DIV
    q_ctr = q_contract(S_OBS, tau, r_q, asof, expiry)
    out = {}
    for lab, qq in (("q = Q_DIV (sheet convention)", q_sheet), ("q = contract-specific", q_ctr)):
        gap = C_OBS - P_OBS - (S_OBS * np.exp(-qq * tau) - K_OBS * np.exp(-r_q * tau))
        base = (S_OBS * np.exp(-qq * tau) - C_OBS + P_OBS) / K_OBS
        r_imp = -np.log(base) / tau
        out[lab] = {"q used": qq, "Parity gap (INR)": gap, "Parity gap (bp of spot)": gap / S_OBS * 1e4,
                    "Implied financing rate": r_imp,
                    "Call implied vol": implied_vol(C_OBS, S_OBS, K_OBS, tau, r_q, qq, "call"),
                    "Put implied vol": implied_vol(P_OBS, S_OBS, K_OBS, tau, r_q, qq, "put")}
    Q4B_TABLE = pd.DataFrame(out)
    print(f"Q4(b): OBSERVED quote | strike {K_OBS:,.2f} | expiry {EXPIRY_OBS} | as of {QUOTE_ASOF} | source: {QUOTE_SOURCE}")
    print(f"       observed call {C_OBS:,.2f}, observed put {P_OBS:,.2f}, spot at the same moment {S_OBS:,.2f}; tau = {tau * 365:.0f} days; "
          f"r = {r_q:.3%} {PROV['inr_r']}")
    display(Q4B_TABLE)
    print(f"Comparison: SIGMA_HAT {SIGMA_HAT:.2%} | latest 21-day vol {VOL21.iloc[-1]:.2%} | inr_r {r_q:.3%} {PROV['inr_r']} | "
          f"NSE-reported IV call {NSE_IV_CALL:.2%}, put {NSE_IV_PUT:.2%}")
    print(f"Quote type: {QUOTE_TYPE}  Volume call/put {VOL_CALL:,}/{VOL_PUT:,} contracts; OI {OI_CALL:,}/{OI_PUT:,}.")
    print(f"Dividends with ex-date between {asof.date()} and {expiry.date()}: "
          f"{int(((DIV_TABLE['ex_date'] > asof) & (DIV_TABLE['ex_date'] <= expiry)).sum())} -> contract-specific q = {q_ctr:.4%}")
    _cdate = stock.index[stock.index <= asof]
    if len(_cdate):
        print(f"Close in the price data on {_cdate[-1].date()}: {stock.loc[_cdate[-1]]:,.2f} {PROV['stock']} (S_OBS = {S_OBS:,.2f})")
    if abs(S_OBS / S0 - 1) > 0.05:
        print("NOTE: S_OBS differs from the last close in the data by more than 5%: check that both are the same stock and date.")


In [ ]:
# ---- YOUR CODE HERE ----  Q4 (c): track the actual call and the synthetic over the last 3 months
def cycle_bounds(idx, n_cycles=3, days=30):
    b = [len(idx) - 1]
    for _ in range(n_cycles):
        b.append(int(idx.searchsorted(idx[b[-1]] - pd.Timedelta(days=days), side="right") - 1))
    return b[::-1]

bounds = cycle_bounds(stock.index)
daily, cyc_rows, cum_prev = [], [], np.zeros(3)
for k in range(3):
    e, x = bounds[k], bounds[k + 1]
    d_e, d_x = stock.index[e], stock.index[x]
    S_e = float(stock.iloc[e]); K_c = S_e
    r_e = float(inr_r.iloc[e]); T_e = (d_x - d_e).days / 365
    sg_e = float(VOL21.iloc[e])
    C_e = bsm_price(S_e, K_c, T_e, r_e, sg_e, Q_DIV, "call"); P_e = bsm_price(S_e, K_c, T_e, r_e, sg_e, Q_DIV, "put")
    L_e = K_c * np.exp(-r_e * T_e)
    for t in range(e, x + 1):
        d_t = stock.index[t]; tau = (d_x - d_t).days / 365
        S_t = float(stock.iloc[t]); r_t = float(inr_r.iloc[t]); sg_t = float(VOL21.iloc[t])
        C_t = bsm_price(S_t, K_c, tau, r_t, sg_t, Q_DIV, "call"); P_t = bsm_price(S_t, K_c, tau, r_t, sg_t, Q_DIV, "put")
        D_t = S_e * (np.exp(Q_DIV * (d_t - d_e).days / 365) - 1)
        L_t = K_c * np.exp(-r_e * tau)
        pnl_call = C_t - C_e
        pnl_syn = S_t + P_t + D_t - S_e - P_e
        pnl_fin = pnl_syn - (L_t - L_e)
        gap_formula = (S_t * (1 - np.exp(-Q_DIV * tau)) + K_c * np.exp(-r_t * tau)
                       - S_e * (1 - np.exp(-Q_DIV * T_e)) - K_c * np.exp(-r_e * T_e) + D_t)
        daily.append({"date": d_t, "cycle": k + 1, "S": S_t, "C": C_t, "P": P_t, "tau_days": (d_x - d_t).days,
                      "pnl_call": pnl_call, "pnl_syn": pnl_syn, "pnl_fin": pnl_fin,
                      "gap_syn_call": pnl_syn - pnl_call, "gap_formula": gap_formula,
                      "gap_fin_call": pnl_fin - pnl_call})
    last = daily[-1]
    cyc_rows.append({"Cycle": k + 1, "Entry": d_e.date(), "Expiry": d_x.date(), "S0": S_e, "S_T": float(stock.iloc[x]),
                     "C0": C_e, "P0": P_e,
                     "P&L call": last["pnl_call"], "P&L synthetic": last["pnl_syn"], "P&L financed": last["pnl_fin"],
                     "ROC call": last["pnl_call"] / C_e, "ROC synthetic": last["pnl_syn"] / (S_e + P_e),
                     "ROC financed": last["pnl_fin"] / (S_e + P_e - L_e)})
trk = pd.DataFrame(daily).set_index("date")
cyc = pd.DataFrame(cyc_rows).set_index("Cycle")

for c in ("pnl_call", "pnl_syn", "pnl_fin"):
    offset, vals = 0.0, []
    for k in (1, 2, 3):
        seg = trk.loc[trk["cycle"] == k, c].to_numpy()
        vals.append(offset + seg); offset += seg[-1]
    trk["cum_" + c] = np.concatenate(vals)

assert np.allclose(trk["gap_syn_call"], trk["gap_formula"], atol=1e-8), (trk["gap_syn_call"] - trk["gap_formula"]).abs().max()
print(f"Q4(c): three cycles ending {stock.index[-1].date()} | stock {PROV['stock']}, r {PROV['inr_r']}, premiums model-generated (21-day vol, no distortion)")
display(cyc)
print("Gap identity checked on every day: synthetic - call = S_t(1-e^(-q tau_t)) + K e^(-r_t tau_t) - S0(1-e^(-qT0)) - K e^(-r0 T0) + D_t")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
ax[0].plot(trk.index, trk["cum_pnl_call"], color="#1f4e79", lw=1.6, label="Actual call")
ax[0].plot(trk.index, trk["cum_pnl_syn"], color="#d17a00", lw=1.6, label="Synthetic: stock + put")
ax[0].plot(trk.index, trk["cum_pnl_fin"], color="#2e7d32", lw=1.6, ls="--", label="Financed synthetic")
for b in bounds[1:3]:
    ax[0].axvline(stock.index[b], color="grey", ls=":", lw=1)
ax[0].set_title(ptitle("Cumulative P&L, three 30-day cycles", premiums=True)); ax[0].set_xlabel("Date"); ax[0].set_ylabel("Cumulative P&L (INR per share)")
ax[0].legend(loc="best")
ax[1].plot(trk.index, trk["gap_syn_call"], color="#d17a00", lw=1.5, label="Synthetic minus call")
ax[1].plot(trk.index, trk["gap_fin_call"], color="#2e7d32", lw=1.5, ls="--", label="Financed synthetic minus call")
ax[1].axhline(0, color="black", lw=0.6)
for b in bounds[1:3]:
    ax[1].axvline(stock.index[b], color="grey", ls=":", lw=1)
ax[1].set_title(ptitle("Gap to the actual call within each cycle", premiums=True)); ax[1].set_xlabel("Date"); ax[1].set_ylabel("P&L difference (INR per share)")
ax[1].legend(loc="best")
plt.tight_layout(); plt.show()


**Your answer (Q4d):**

*Data provenance — which numbers above are observed and which are model-generated:*

*Which portfolio replicates the call; what the profit gap between (1) and (2) is (formula and ₹):*

*What the synthetic-minus-call gap equals on any day within a cycle:*

*Is the observed parity gap tradable? Which arbitrage, and three obstacles:*

*Implied vs realised volatility:*

---
## Supplement A · Which volatility estimate forecasts best?

Nine estimators, each using returns up to the forecast date only, scored against the volatility realised over the next 30 days. Q5 keeps the sheet's 21-day estimate; all estimators tried are shown.

In [ ]:
# ==================================================================================
#  SUPPLEMENT A: which volatility estimate predicts best?  (my addition, not asked for by the sheet)
# ==================================================================================
from scipy.signal import lfilter

def garch11_fit(r_pct):
    """Gaussian quasi-ML fit of a GARCH(1,1) on returns in percent (zero mean). Returns omega, alpha, beta."""
    v0 = float(np.var(r_pct)); r2 = r_pct ** 2
    def nll(p):
        om, al, be = p
        x = om + al * r2[:-1]
        h = np.concatenate([[v0], lfilter([1.0], [1.0, -be], x, zi=[be * v0])[0]])
        return 0.5 * np.sum(np.log(h) + r2 / h)
    x0 = [v0 * 0.05, 0.08, 0.90]
    res = optimize.minimize(nll, x0, method="SLSQP", bounds=[(1e-8, None), (0.0, 1.0), (0.0, 1.0)],
                            constraints=[{"type": "ineq", "fun": lambda p: 0.999 - p[1] - p[2]}])
    return res.x if res.success else np.array(x0)

def garch11_vol(r_full, refit_every=21, min_obs=100, horizon=21):
    """Expanding-window GARCH(1,1), refitted every 21 days. The forecast on date t is the average annualised vol over"""
    r = r_full.dropna(); v = r.to_numpy() * 100; n = len(v); out = np.full(n, np.nan)
    for start in range(min_obs, n, refit_every):
        om, al, be = garch11_fit(v[:start])
        end = min(start + refit_every, n)
        v0 = float(np.var(v[:start])); r2 = v[:end] ** 2
        h = np.concatenate([[v0], lfilter([1.0], [1.0, -be], om + al * r2[:-1], zi=[be * v0])[0]])
        h_next = om + al * r2 + be * h
        vbar = om / max(1 - al - be, 1e-6)
        k = np.arange(horizon)
        for t in range(start, end):
            avg_var = vbar + ((al + be) ** k * (h_next[t] - vbar)).mean()
            out[t] = np.sqrt(max(avg_var, 1e-12) * 252) / 100
    return pd.Series(out, index=r.index).reindex(r_full.index)

EST = pd.DataFrame(index=stock.index)
EST["RV21"]   = ret_all.rolling(21).std() * np.sqrt(252)
EST["RV63"]   = ret_all.rolling(63).std() * np.sqrt(252)
EST["RV126"]  = ret_all.rolling(126).std() * np.sqrt(252)
EST["EXP"]    = ret_all.expanding(min_periods=60).std() * np.sqrt(252)
for h in (10, 21, 63):
    EST[f"EWMA_h{h}"] = np.sqrt((ret_all ** 2).ewm(halflife=h, adjust=False, min_periods=30).mean() * 252)
EST["BLEND"]  = EST[["RV21", "RV63", "EXP"]].mean(axis=1, skipna=False)
EST["GARCH11"] = garch11_vol(ret_all)
EST_NAMES = list(EST.columns)

valid_e, pos_e = expiry_map(stock.index, 30)
TARGET = pd.Series(fwd_rv(ret_all, pos_e), index=stock.index)
TARGET[~valid_e] = np.nan

ev = (stock.index >= pd.Timestamp(WIN_START)) & valid_e & EST.notna().all(axis=1).to_numpy() & TARGET.notna().to_numpy()
E, Y = EST.loc[ev], TARGET.loc[ev]
print(f"Evaluation sample {PROV['stock']}: {len(E)} entry dates from {E.index[0].date()} to {E.index[-1].date()} "
      f"(every estimator available; overlapping 30-day targets, so far fewer than {len(E)} independent points)")

def qlike(f, y):
    ratio = (y / f) ** 2
    return float(np.mean(ratio - np.log(ratio) - 1))

atm = lambda s, rr: bsm_price(1.0, 1.0, 30 / 365, rr, s, 0.0, "call")
rr_e = inr_r.loc[E.index].to_numpy()
rows = []
no = np.arange(0, len(E), 21)
for nm in EST_NAMES:
    f = E[nm].to_numpy(); y = Y.to_numpy()
    slope, icpt = np.polyfit(f, y, 1)
    prem_err = np.mean(np.abs(atm(f, rr_e) - atm(y, rr_e)) / atm(y, rr_e))
    rows.append({"Estimator": nm, "Bias (vol pts)": np.mean(f - y) * 100, "RMSE (vol pts)": np.sqrt(np.mean((f - y) ** 2)) * 100,
                 "MAE (vol pts)": np.mean(np.abs(f - y)) * 100, "QLIKE": qlike(f, y),
                 "MZ slope": slope, "MZ R2": np.corrcoef(f, y)[0, 1] ** 2,
                 "ATM premium error (%)": prem_err * 100,
                 "RMSE every 21st (vol pts)": np.sqrt(np.mean((f[no] - y[no]) ** 2)) * 100, "QLIKE every 21st": qlike(f[no], y[no])})
VOL_STUDY = pd.DataFrame(rows).set_index("Estimator")
VOL_STUDY["Mean rank"] = VOL_STUDY[["RMSE (vol pts)", "MAE (vol pts)", "QLIKE", "ATM premium error (%)"]].rank().mean(axis=1)
print("All-date metrics use overlapping 30-day targets; the 'every 21st' columns use non-overlapping ones.")
display(VOL_STUDY.sort_values("QLIKE"))

d_rows = []
for nm in EST_NAMES:
    if nm == "RV21":
        continue
    dl = ((Y.to_numpy()[no] - E["RV21"].to_numpy()[no]) ** 2) - ((Y.to_numpy()[no] - E[nm].to_numpy()[no]) ** 2)
    d_rows.append({"Estimator": nm, "Mean squared-error gain over RV21 (vol pts^2)": dl.mean() * 1e4,
                   "t-stat (every 21st date)": dl.mean() / (dl.std(ddof=1) / np.sqrt(len(dl))), "n": len(dl)})
display(pd.DataFrame(d_rows).set_index("Estimator"))

BEST_VOL_EST = VOL_STUDY["QLIKE"].idxmin()
print(f"Lowest QLIKE: {BEST_VOL_EST}. Lowest RMSE: {VOL_STUDY['RMSE (vol pts)'].idxmin()}. Best mean rank: {VOL_STUDY['Mean rank'].idxmin()}.")
print("This ranking is in-sample: the same two years are used to pick the estimator and, later, to run Q5. "
      f"{len(EST_NAMES)} estimators were tried and all are shown.")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
for nm, c in [("RV21", "#d17a00"), ("EWMA_h21", "#1f4e79"), ("BLEND", "#2e7d32"), ("GARCH11", "#7b3fa0")]:
    ax[0].plot(E.index, E[nm] * 100, color=c, lw=1.1, label=nm)
ax[0].plot(Y.index, Y * 100, color="black", lw=1.4, label="Realised over next 30 days")
ax[0].set_title(ptitle("Volatility forecasts against what followed")); ax[0].set_xlabel("Entry date"); ax[0].set_ylabel("Annualised volatility (% p.a.)")
ax[0].legend(loc="best", fontsize=8)
order = VOL_STUDY["QLIKE"].sort_values()
ax[1].barh(order.index[::-1], order.values[::-1], color="#1f4e79", label="QLIKE (lower is better)")
ax[1].set_title(ptitle("Forecast loss by estimator")); ax[1].set_xlabel("QLIKE loss (dimensionless)"); ax[1].set_ylabel("Estimator")
ax[1].legend(loc="best")
plt.tight_layout(); plt.show()


---
## Q5 · Historical arbitrage simulation: synthetic vs actual call · **[8 marks]**

Repeat the comparison of Q4 on **every trading day** of the two-year window.

**Fixed parameters.** Time to expiry: 30 calendar days. Strike: at the money ($K$ = that day's
close). Horizon: two years of daily data. Each day you open two positions and hold both to expiry:

- **Strategy A** — buy the actual call.
- **Strategy B** — synthetic call: buy the stock and buy the put.

**(a) Build the trade panel** with one row per entry date and columns
`S0, K, r, T, expiry, S_T, sigma_call, sigma_put, C0, P0, pnl_call, pnl_syn, pnl_fin, cap_call, cap_syn`,
where `pnl_fin` is the P&L of the financed synthetic of Q4 and `cap_*` is the capital paid on day 0.
Rules:
- the expiry is the last trading day on or before entry + 30 calendar days; drop entry dates whose
  expiry lies beyond your data, and report how many trades remain (it will be a little under 500);
- **no look-ahead** — the volatility used on a date may only use returns up to that date;
- the closes are not dividend-adjusted, so credit the stock leg of strategy B with dividend income
  of $S_0(e^{qT}-1)$ over the holding period, using your $q$ from Q2;
- if you have real historical option premiums (NSE bhavcopy, `jugaad-data`, `nsepython`), use them
  and say so. If not, generate $C_0$ and $P_0$ from your own pricer with $\sigma_t$ = the 21-day
  realised volatility, and add a **documented mean-reverting distortion** to the put's volatility to
  stand in for the parity gaps seen in real markets — **and state clearly in your write-up that the
  option premiums are synthetic.** Make every parameter of the distortion a named constant.

**(b) Performance comparison table**, per share, in exactly this layout:

| Metric | Actual Call | Synthetic Call | Winner |
|---|---|---|---|
| Total trades profitable | X / N | Y / N | |
| Avg. return per trade | ₹Z | ₹W | |
| Max single-trade profit | ₹P | ₹Q | |
| Max single-trade loss | ₹R | ₹S | |
| Avg. capital required | ₹ | ₹ | |
| Avg. return on capital (ROC) | % | % | |
| Std. dev. of ROC | % | % | |

Then report, for the **financed** synthetic against the call: the mean and standard deviation of
the P&L difference, the share of days on which the absolute difference exceeds a transaction-cost
band of **25 bps of spot**. Finally report a $t$-statistic for (i) the mean P&L of the actual call
and (ii) the mean financed-minus-call difference, **each computed two ways** — on all trades, and on
a **non-overlapping** subsample (every 21st trade).

**(c) Best and worst cases, and cost efficiency.** List the entry dates of the maximum profit and
maximum loss of each strategy, with the stock's move over the holding period, the volatility
priced in at entry, and the volatility **realised** over the holding period. Produce three
diagnostic plots:
1. P&L per trade of both strategies through time;
2. P&L against the stock's percentage move over the holding period (both strategies);
3. the distribution of **return on capital** for both strategies.

**(d)** Write **at least 150 words** answering all four of the following:
- Synthetic "wins" on rupee P&L. By how much per trade on average, what is that amount in
  economic terms, and what happens to it once the stock leg is financed? **Is anything left that
  you would call an arbitrage?**
- Which strategy wins on return on capital, and why is that the mirror image of the rupee result?
  Which would you rather hold if you had ₹1 lakh and a one-month bullish view?
- What do the best and worst trades have in common? Relate them to the stock's move and to
  realised-minus-priced volatility.
- You have nearly 500 trades. How many **independent** observations do you really have, and what
  does that do to any claim of statistical significance? List at least three costs or frictions
  your simulation still ignores.

| Part | Rubric — what earns the marks | Marks |
|---|---|---|
| (a) | Correct expiry mapping and trade count; volatility without look-ahead; P&L and capital definitions correct for all three portfolios; premium source documented (real or synthetic with named distortion parameters) | 2 |
| (b) | Table in the required layout with a justified "winner" in each row; financed comparison with cost band; $t$-statistics on all trades and on the non-overlapping subsample | 2 |
| (c) | Best/worst dates tabulated with move, priced and realised volatility; three diagnostic plots, labelled | 2 |
| (d) | ≥150 words; rupee advantage identified as financing cost ($\approx S_0(r-q)T$) and shown to vanish after financing, leaving only the distortion; leverage explains the ROC reversal; best/worst trades tied to large moves and to realised vs priced volatility; overlap reduces ~500 trades to ~24 independent ones; three or more ignored frictions | 2 |

In [ ]:
# ---- YOUR CODE HERE ----  Q5 (a)
# No historical option prices available: C0 and P0 are model-generated (21-day vol, no look-ahead).
# Put vol = call vol + x_t, with x_t an AR(1) distortion standing in for real parity gaps:
#   x_t = DIST_MU + DIST_PHI (x_{t-1} - DIST_MU) + DIST_SD z_t,  z_t ~ N(0, 1)
# 1 vol point on a 30-day ATM option is about 11.5 bp of spot, so these values give gaps of the order of the 25 bp cost band.
# The values were fixed before looking at the results and are not tuned.
DIST_MU    = 0.010   # mean add-on, 1 vol point (puts rich)
DIST_PHI   = 0.90    # daily persistence, half-life 6.6 days
DIST_SD    = 0.006   # daily shock sd; stationary sd 1.4 points
DIST_FLOOR = 0.05    # put vol floor
DIST_SEED  = SEED + 81

def build_trades(spot, rate, q, horizon_days=30, vol_window=21, seed=SEED + 81, vol_series=None, start=None,
                 dist_mu=DIST_MU, dist_phi=DIST_PHI, dist_sd=DIST_SD, dist_floor=DIST_FLOOR, div=DIV_TABLE):
    """One row per entry date: buy the actual call (A) and the synthetic stock + put (B), hold both to expiry."""
    start = WIN_START if start is None else start
    idx = spot.index
    lr = np.log(spot).diff()
    vol = lr.rolling(vol_window).std() * np.sqrt(252) if vol_series is None else vol_series.reindex(idx)
    valid, pos = expiry_map(idx, horizon_days)
    rng = np.random.default_rng(seed); x = np.empty(len(idx)); prev = dist_mu
    for i in range(len(idx)):
        prev = dist_mu + dist_phi * (prev - dist_mu) + dist_sd * rng.standard_normal(); x[i] = prev
    keep = valid & (idx >= pd.Timestamp(start)) & vol.notna().to_numpy()
    ei = np.where(keep)[0]; xi = pos[ei]
    q = q.reindex(idx).to_numpy()[ei] if isinstance(q, pd.Series) else q
    S0_, ST_ = spot.to_numpy()[ei], spot.to_numpy()[xi]
    K_ = S0_.copy()
    r_ = rate.reindex(idx).to_numpy()[ei]
    T_ = np.array([(idx[b] - idx[a]).days for a, b in zip(ei, xi)]) / 365
    sg_c = vol.to_numpy()[ei]
    sg_p = np.maximum(sg_c + x[ei], dist_floor)
    C0 = bsm_price(S0_, K_, T_, r_, sg_c, q, "call"); P0 = bsm_price(S0_, K_, T_, r_, sg_p, q, "put")
    Dc = S0_ * (np.exp(q * T_) - 1)   # dividend credit on the stock leg
    loan = K_ * np.exp(-r_ * T_)
    pnl_call = np.maximum(ST_ - K_, 0) - C0
    pnl_syn = (ST_ - S0_) + np.maximum(K_ - ST_, 0) - P0 + Dc
    pnl_fin = pnl_syn - (K_ - loan)   # borrow K e^(-rT) at entry, repay K at expiry
    ex = div["ex_date"].to_numpy()[None, :]; amt = div["amount"].to_numpy()[None, :]
    inwin = (ex > idx.to_numpy()[ei][:, None]) & (ex <= idx.to_numpy()[xi][:, None])
    tr = pd.DataFrame({
        "S0": S0_, "K": K_, "r": r_, "T": T_, "expiry": idx[xi], "S_T": ST_, "sigma_call": sg_c, "sigma_put": sg_p,
        "C0": C0, "P0": P0, "pnl_call": pnl_call, "pnl_syn": pnl_syn, "pnl_fin": pnl_fin,
        "cap_call": C0, "cap_syn": S0_ + P0,
        "cap_fin": S0_ + P0 - loan, "roc_call": pnl_call / C0, "roc_syn": pnl_syn / (S0_ + P0), "roc_fin": pnl_fin / (S0_ + P0 - loan),
        "move": ST_ / S0_ - 1, "rv_hold": fwd_rv(lr, pos)[ei], "div_credit": Dc, "div_cash": (inwin * amt).sum(axis=1),
        "has_special": (inwin & (div["kind"].to_numpy()[None, :] == "special")).any(axis=1),
        "dist_x": x[ei], "parity_gap": C0 - P0 - (S0_ * np.exp(-q * T_) - K_ * np.exp(-r_ * T_)),
    }, index=idx[ei])
    tr.index.name = "entry"
    return tr

trades = build_trades(stock, inr_r, Q_DIV)
print(f"Q5(a): {len(trades)} trades remain after dropping entries whose expiry lies beyond the data "
      f"(window days {int((stock.index >= pd.Timestamp(WIN_START)).sum())}, dropped {int((stock.index >= pd.Timestamp(WIN_START)).sum()) - len(trades)})")
print(f"Underlying {PROV['stock']}; rate {PROV['inr_r']}; option premiums MODEL-GENERATED (21-day vol; put vol distorted); q = {Q_DIV:.3%}")
print(f"Distortion constants: DIST_MU={DIST_MU}, DIST_PHI={DIST_PHI}, DIST_SD={DIST_SD}, DIST_FLOOR={DIST_FLOOR}, seed={DIST_SEED}")
assert trades.index.is_monotonic_increasing and (trades["expiry"] > trades.index).all()
assert trades.isna().sum().sum() == 0
assert ((trades["expiry"] - trades.index).dt.days <= 30).all() and ((trades["expiry"] - trades.index).dt.days >= 26).all()
_probe = stock.copy(); _cut = trades.index[len(trades) // 2]
_probe.loc[_probe.index > _cut] = _probe.loc[_probe.index > _cut] * np.exp(np.random.default_rng(0).normal(0, 0.3, (_probe.index > _cut).sum()))
_vol_a = (np.log(stock).diff().rolling(21).std() * np.sqrt(252)).loc[_cut]; _vol_b = (np.log(_probe).diff().rolling(21).std() * np.sqrt(252)).loc[_cut]
assert abs(_vol_a - _vol_b) < 1e-12, "look-ahead detected"
_alg = trades["div_credit"] - trades["S0"] * (1 - np.exp(-Q_DIV * trades["T"])) + trades["parity_gap"]
assert np.allclose(trades["pnl_fin"] - trades["pnl_call"], _alg, atol=1e-8)
print("Checks passed: expiry mapping, no NaN, no look-ahead, and pnl_fin - pnl_call = D - S0(1-e^(-qT)) + parity_gap exactly.")
display(trades[["S0", "K", "r", "T", "expiry", "S_T", "sigma_call", "sigma_put", "C0", "P0", "pnl_call", "pnl_syn", "pnl_fin", "cap_call", "cap_syn"]].head(5))


In [ ]:
# ---- YOUR CODE HERE ----  Q5 (b): performance comparison table, financed comparison, t-statistics
TC_BPS = 25.0

def _inr(v):  return f"₹{v:,.2f}"
def _pct(v):  return f"{v:.2%}"

n_tr = len(trades)
A, B = trades["pnl_call"], trades["pnl_syn"]
roc_A, roc_B = trades["roc_call"], trades["roc_syn"]

rows = [
    ("Total trades profitable",       f"{int((A > 0).sum())} / {n_tr}", f"{int((B > 0).sum())} / {n_tr}", "Actual Call" if (A > 0).sum() > (B > 0).sum() else "Synthetic Call"),
    ("Avg. return per trade",         _inr(A.mean()),  _inr(B.mean()),  "Actual Call" if A.mean() > B.mean() else "Synthetic Call"),
    ("Max single-trade profit",       _inr(A.max()),   _inr(B.max()),   "Actual Call" if A.max() > B.max() else "Synthetic Call"),
    ("Max single-trade loss",         _inr(A.min()),   _inr(B.min()),   "Actual Call" if A.min() > B.min() else "Synthetic Call"),
    ("Avg. capital required",         _inr(trades["cap_call"].mean()), _inr(trades["cap_syn"].mean()),
                                      "Actual Call" if trades["cap_call"].mean() < trades["cap_syn"].mean() else "Synthetic Call"),
    ("Avg. return on capital (ROC)",  _pct(roc_A.mean()), _pct(roc_B.mean()), "Actual Call" if roc_A.mean() > roc_B.mean() else "Synthetic Call"),
    ("Std. dev. of ROC",              _pct(roc_A.std(ddof=1)), _pct(roc_B.std(ddof=1)), "Actual Call" if roc_A.std(ddof=1) < roc_B.std(ddof=1) else "Synthetic Call"),
]
Q5_TABLE = pd.DataFrame(rows, columns=["Metric", "Actual Call", "Synthetic Call", "Winner"]).set_index("Metric")
print(f"Q5(b): per share, {n_tr} trades; underlying {PROV['stock']}; option premiums MODEL-GENERATED (not market data)")
display(Q5_TABLE)
print("Winner rule: more profitable trades, higher mean P&L, larger maximum profit, SMALLER maximum loss, LESS capital, "
      "higher mean ROC, LOWER ROC standard deviation.")

diff = trades["pnl_fin"] - trades["pnl_call"]
band = TC_BPS / 1e4 * trades["S0"]
share_out = float((diff.abs() > band).mean())
print(f"\nFinanced synthetic minus actual call (INR per share): mean {diff.mean():.4f}, sd {diff.std(ddof=1):.4f}, "
      f"min {diff.min():.4f}, max {diff.max():.4f}")
print(f"Share of trades with |difference| > {TC_BPS:.0f} bp of spot: {share_out:.1%}  (mean band {band.mean():.2f} INR)")
fin_cost = trades["K"] * (1 - np.exp(-trades["r"] * trades["T"]))
wedge = trades["S0"] * (1 - np.exp(-Q_DIV * trades["T"]))
decomp = pd.Series({"financing cost K(1 - e^(-rT))": fin_cost.mean(), "+ dividend credit D = S0(e^(qT) - 1)": trades["div_credit"].mean(),
                    "- dividend wedge S0(1 - e^(-qT))": -wedge.mean(), "+ parity gap (put-volatility distortion)": trades["parity_gap"].mean()})
decomp["= synthetic minus call"] = decomp.sum()
assert abs(decomp["= synthetic minus call"] - (B - A).mean()) < 1e-8
print("\nWhere the synthetic's rupee edge over the call comes from (INR per share, mean over trades):")
display(decomp.to_frame("INR per share"))
print(f"For reference, S0 (r - q) T averaged over trades = {(trades['S0'] * (trades['r'] - Q_DIV) * trades['T']).mean():.4f} INR. "
      "The edge equals S0 (r - q) T only when the stock leg is NOT given the dividend credit; with the credit (rule 4) the dividend terms cancel "
      f"and the edge is about S0 r T = {(trades['S0'] * trades['r'] * trades['T']).mean():.4f} INR plus the parity gap.")

def tstat(x):
    x = np.asarray(x, dtype=float)
    return float(x.mean() / (x.std(ddof=1) / np.sqrt(len(x))))

def nw_tstat(x, lags=21):
    """Newey-West t-statistic of the mean (Bartlett weights), the proper fix for 30-day overlapping trades."""
    x = np.asarray(x, dtype=float); n = len(x); e = x - x.mean()
    lrv = e @ e / n
    for L in range(1, lags + 1):
        lrv += 2 * (1 - L / (lags + 1)) * (e[L:] @ e[:-L]) / n
    return float(x.mean() / np.sqrt(lrv / n))

tt = []
for lab, x in (("Mean P&L of the actual call", A), ("Mean financed-minus-call difference", diff)):
    offs = [tstat(x.to_numpy()[o::21]) for o in range(21)]
    tt.append({"Quantity": lab, "t, all trades": tstat(x), "n all": len(x),
               "t, every 21st (offset 0)": tstat(x.to_numpy()[::21]), "n every 21st": len(x.to_numpy()[::21]),
               "t, every 21st: min over 21 offsets": min(offs), "t, every 21st: max over 21 offsets": max(offs),
               "Newey-West t (21 lags), all trades": nw_tstat(x)})
Q5_TSTATS = pd.DataFrame(tt).set_index("Quantity")
display(Q5_TSTATS)


In [ ]:
# ---- YOUR CODE HERE ----  Q5 (c): best / worst trades and the three diagnostic plots
cases = []
for strat, col in (("Actual call", "pnl_call"), ("Synthetic (stock + put)", "pnl_syn")):
    for case, d in (("Maximum profit", trades[col].idxmax()), ("Maximum loss", trades[col].idxmin())):
        t = trades.loc[d]
        cases.append({"Strategy": strat, "Case": case, "Entry": d.date(), "Expiry": t["expiry"].date(),
                      "Stock move over hold": t["move"], "Priced-in vol at entry (call)": t["sigma_call"],
                      "Priced-in vol (put, distorted)": t["sigma_put"], "Realised vol over hold": t["rv_hold"],
                      "Realised minus priced": t["rv_hold"] - t["sigma_call"], "P&L (INR)": t[col]})
Q5C_TABLE = pd.DataFrame(cases).set_index(["Strategy", "Case"])
print(f"Q5(c): best and worst trades, underlying {PROV['stock']}, premiums model-generated")
display(Q5C_TABLE.style.format({"Stock move over hold": "{:.2%}", "Priced-in vol at entry (call)": "{:.2%}", "Priced-in vol (put, distorted)": "{:.2%}",
                                "Realised vol over hold": "{:.2%}", "Realised minus priced": "{:.2%}", "P&L (INR)": "{:,.2f}"}))

fig, ax = plt.subplots(1, 3, figsize=(17, 4.8))
ax[0].plot(trades.index, trades["pnl_call"], color="#1f4e79", lw=1.0, label="Actual call")
ax[0].plot(trades.index, trades["pnl_syn"], color="#d17a00", lw=1.0, label="Synthetic: stock + put")
ax[0].axhline(0, color="black", lw=0.6)
ax[0].set_title(ptitle("P&L per trade by entry date", premiums=True)); ax[0].set_xlabel("Entry date"); ax[0].set_ylabel("P&L (INR per share)")
ax[0].legend(loc="best", fontsize=8)
ax[1].scatter(trades["move"] * 100, trades["pnl_call"], s=9, color="#1f4e79", alpha=0.6, label="Actual call")
ax[1].scatter(trades["move"] * 100, trades["pnl_syn"], s=9, color="#d17a00", alpha=0.6, label="Synthetic: stock + put")
ax[1].axhline(0, color="black", lw=0.6); ax[1].axvline(0, color="black", lw=0.6)
ax[1].set_title(ptitle("P&L against the stock's move over the hold", premiums=True)); ax[1].set_xlabel("Stock price change from entry to expiry (%)")
ax[1].set_ylabel("P&L (INR per share)"); ax[1].legend(loc="best", fontsize=8)
bins = np.linspace(min(trades["roc_call"].min(), trades["roc_syn"].min()), max(trades["roc_call"].max(), trades["roc_syn"].max()), 50) * 100
ax[2].hist(trades["roc_call"] * 100, bins=bins, alpha=0.6, color="#1f4e79", label="Actual call")
ax[2].hist(trades["roc_syn"] * 100, bins=bins, alpha=0.6, color="#d17a00", label="Synthetic: stock + put")
ax[2].set_title(ptitle("Distribution of return on capital", premiums=True)); ax[2].set_xlabel("Return on capital per trade (%)"); ax[2].set_ylabel("Number of trades")
ax[2].legend(loc="best", fontsize=8)
plt.tight_layout(); plt.show()

print(f"Mean synthetic minus call: {(B - A).mean():.4f} INR = {((B - A) / trades['S0']).mean() * 1e4:.1f} bp of spot; "
      f"mean financing cost K(1-e^(-rT)) = {(trades['K'] * (1 - np.exp(-trades['r'] * trades['T']))).mean():.4f} INR; "
      f"mean dividend credit {trades['div_credit'].mean():.4f} INR")
print(f"After financing: mean {diff.mean():.4f} INR; mean parity gap (the put distortion) {trades['parity_gap'].mean():.4f} INR = "
      f"{(trades['parity_gap'] / trades['S0']).mean() * 1e4:.1f} bp of spot")
print(f"Effective independent trades: {n_tr} trades / about 21 trading days per 30-day cycle = about {n_tr / 21:.0f}")


---
## Supplement C · Robustness of Q5

Q5 re-run under each volatility estimator, each treatment of $q$, 200 seeds of the put-volatility distortion, and each half of the window.

In [ ]:
# ==================================================================================
#  SUPPLEMENT C: does the Q5 conclusion depend on the volatility estimator or on the treatment of q?
# ==================================================================================
common_idx = E.index
rows = []
for nm in EST_NAMES:
    tr = build_trades(stock, inr_r, Q_DIV, vol_series=EST[nm]); tr = tr.loc[tr.index.intersection(common_idx)]
    d = tr["pnl_fin"] - tr["pnl_call"]
    rows.append({"Volatility estimator": nm, "Trades": len(tr), "Mean sigma_call": tr["sigma_call"].mean(),
                 "Mean call P&L (INR)": tr["pnl_call"].mean(), "t, all": tstat(tr["pnl_call"]), "t, every 21st": tstat(tr["pnl_call"].to_numpy()[::21]),
                 "Mean ROC call": tr["roc_call"].mean(), "Mean ROC synthetic": tr["roc_syn"].mean(),
                 "Mean financed - call (INR)": d.mean(), "Share beyond 25 bp": (d.abs() > TC_BPS / 1e4 * tr["S0"]).mean()})
SENS_VOL = pd.DataFrame(rows).set_index("Volatility estimator")
print(f"Supplement C(1): Q5 re-run on the common sample of {len(common_idx)} entry dates; underlying {PROV['stock']}; premiums model-generated")
display(SENS_VOL)

rows = []
# causal q: dividends in the 365 days up to each date / that day's close (no future dividends or prices)
Q_CAUSAL = pd.Series([DIV_TABLE.loc[(DIV_TABLE["ex_date"] > d - pd.Timedelta(days=365)) & (DIV_TABLE["ex_date"] <= d), "amount"].sum() / s
                      for d, s in stock.items()], index=stock.index)
for nm, qq in (("q = 0", 0.0), ("TTM regular only", Q_REGULAR), ("TTM incl. special", Q_TOTAL), ("Causal rolling TTM", Q_CAUSAL)):
    tr = build_trades(stock, inr_r, qq); d = tr["pnl_fin"] - tr["pnl_call"]
    rows.append({"Treatment of q": nm, "q": float(np.mean(qq)), "Trades": len(tr), "Mean synthetic - call (INR)": (tr["pnl_syn"] - tr["pnl_call"]).mean(),
                 "Mean S0 (r - q) T (INR)": (tr["S0"] * (tr["r"] - (qq.reindex(tr.index) if isinstance(qq, pd.Series) else qq)) * tr["T"]).mean(), "Mean financed - call (INR)": d.mean(),
                 "Mean parity gap (INR)": tr["parity_gap"].mean(), "Mean call P&L (INR)": tr["pnl_call"].mean(),
                 "Mean ROC call": tr["roc_call"].mean(), "Mean ROC synthetic": tr["roc_syn"].mean(),
                 "Share beyond 25 bp": (d.abs() > TC_BPS / 1e4 * tr["S0"]).mean()})
SENS_Q = pd.DataFrame(rows).set_index("Treatment of q")
print("\nSupplement C(2): Q5 re-run with each treatment of q (volatility = RV21)")
display(SENS_Q)
# distortion is one random path: repeat with other seeds
seed_rows = []
for sd in range(SEED, SEED + 200):
    tr = build_trades(stock, inr_r, Q_DIV, seed=sd); d = tr["pnl_fin"] - tr["pnl_call"]
    seed_rows.append({"mean": d.mean(), "share_25bp": (d.abs() > TC_BPS / 1e4 * tr["S0"]).mean(),
                      "t_all": tstat(d), "t_21": tstat(d.to_numpy()[::21])})
SEEDS = pd.DataFrame(seed_rows)
print("\nSupplement C(3): financed - call over 200 distortion seeds (same constants)")
display(SEEDS.describe(percentiles=[0.05, 0.5, 0.95]).loc[["mean", "std", "5%", "50%", "95%"]])

# out-of-sample style check: does the headline hold in each half of the window separately?
half = trades.index[len(trades) // 2]
split = []
for lab, tr in (("first half", trades.loc[:half]), ("second half", trades.loc[half:].iloc[1:])):
    d = tr["pnl_fin"] - tr["pnl_call"]
    split.append({"Sample": lab, "From": tr.index[0].date(), "To": tr.index[-1].date(), "Trades": len(tr),
                  "Mean call P&L": tr["pnl_call"].mean(), "t call (every 21st)": tstat(tr["pnl_call"].to_numpy()[::21]),
                  "Mean syn - call": (tr["pnl_syn"] - tr["pnl_call"]).mean(), "Mean financed - call": d.mean(),
                  "Mean move": tr["move"].mean(), "Mean priced vol": tr["sigma_call"].mean(), "Mean realised vol": tr["rv_hold"].mean()})
SPLIT = pd.DataFrame(split).set_index("Sample")
print("\nSupplement C(4): Q5 headline in each half of the window")
display(SPLIT)


---
## Numbers used in the written answers

In [ ]:
# ==================================================================================
#  HELPER (my addition): one-screen digest of the numbers the written answers quote.
#  Every value below is printed from variables computed above; nothing is typed in.
# ==================================================================================
def _p(label, value):
    print(f"  {label:<62s} {value}")

print(f"DIGEST  {TICKER}  | stock {PROV['stock']}  | rate {PROV['inr_r']}  | data to {stock.index[-1].date()}  | window from {WIN_START}")
print("Q1");  _p("SIGMA_HAT", f"{SIGMA_HAT:.2%}")
_p("21-day vol min / max / latest", f"{v21.min():.2%} ({v21.idxmin().date()}) / {v21.max():.2%} ({v21.idxmax().date()}) / {v21.iloc[-1]:.2%}")
_p("63-day vol latest", f"{VOL63.iloc[-1]:.2%}")
_p("Total return, ann. mean log return, max drawdown", f"{Q1_STATS['total_return']:.2%}, {Q1_STATS['ann_drift']:.2%}, {Q1_STATS['max_dd']:.2%}")
_p("Skew, excess kurtosis, JB p-value", f"{Q1_STATS['skew']:.3f}, {Q1_STATS['excess_kurt']:.3f}, {jb_pvalue(ret):.3g}")
_p("ACF of squared returns, lags 1-5", ", ".join(f"{a:.2f}" for a in acf_sq))
_p("Days flagged by the 8% screen", f"{len(flag_table)}")
print("Q2");  _p("S0, r, q (treatment)", f"{S0:,.2f}, {R0:.3%}, {Q_DIV:.3%} ({DIV_TREATMENT}; regular only {Q_REGULAR:.3%})")
_p("ATM call / put (model)", f"{C_ATM:,.2f} / {P_ATM:,.2f}")
_p("Call delta, put delta, gamma", f"{Q2_TABLE.iloc[0, 1]:.4f}, {Q2_TABLE.iloc[1, 1]:.4f}, {Q2_TABLE.iloc[0, 2]:.5f}")
_p("Vega per vol point, theta per day (call / put)", f"{Q2_TABLE.iloc[0, 3]:.3f}, {Q2_TABLE.iloc[0, 4]:.3f} / {Q2_TABLE.iloc[1, 4]:.3f}")
_p("d1 at the money, N(d1)", f"{d1_atm:.4f}, {norm.cdf(d1_atm):.4f}")
_p("ATM gamma at 1 week vs 1 month (ratio)", f"{bsm_greeks(S0, K0, 7/365, R0, SIG0, Q_DIV)['gamma'] / bsm_greeks(S0, K0, 30/365, R0, SIG0, Q_DIV)['gamma']:.2f}x")
print("Q3");  _p("Tree minus BSM, call N=50 / N=500 (INR)", f"{Q3_TABLE.loc[('ATM call', 50), 'Difference (INR)']:.4f} / {Q3_TABLE.loc[('ATM call', 500), 'Difference (INR)']:.4f}")
_p("Fitted log-log slope (ATM), c in c/N, N for 0.01 INR", f"{conv_slope:.2f}, {c_ref:.2f}, about {int(np.ceil(c_ref / 0.01)):,}")
_p("ATM error N=50/51, 100/101", f"{err_atm[49]:+.4f}/{err_atm[50]:+.4f}, {err_atm[99]:+.4f}/{err_atm[100]:+.4f}")
_p("Mean |ATM error| N=50..400 raw vs averaged", f"{np.mean(np.abs(err_atm[49:])):.4f} vs {np.mean(np.abs(avg_atm[49:] - bs_atm)):.4f}")
_p("Mean |OTM error| N=50..400 raw vs averaged", f"{np.mean(np.abs(err_otm[49:])):.4f} vs {np.mean(np.abs(avg_otm[49:] - bs_otm)):.4f}")
_p("Tree parity gap at N=5", f"{abs(binomial_price(S0, K0, T0, R0, SIG0, 5, Q_DIV, 'call') - binomial_price(S0, K0, T0, R0, SIG0, 5, Q_DIV, 'put') - parity_rhs):.2e}")
print("Q4(a)"); _p("Costs (1) call, (2) stock+put, (3) financed", f"{cost['(1) Actual call']:,.2f}, {cost['(2) Synthetic: stock + put']:,.2f}, {cost['(3) Financed synthetic: stock + put - loan']:,.2f}")
_p("Loan K e^(-rT), dividend credit D, dividend wedge", f"{loan:,.2f}, {D0:.2f}, {S0 * (1 - np.exp(-Q_DIV * T0)):.2f}")
_p("Profit gap (2)-(1) = K + D - S0 - P0 + C0", f"{K0 + D0 - S0 - P_ATM + C_ATM:.2f}  (financing K(1-e^-rT) = {K0 * (1 - np.exp(-R0 * T0)):.2f})")
print("Q4(b)")
if QUOTE_READY:
    for col in Q4B_TABLE.columns:
        t = Q4B_TABLE[col]
        _p(f"[{col}]", f"q {t['q used']:.3%} | gap {t['Parity gap (INR)']:+.2f} INR = {t['Parity gap (bp of spot)']:+.1f} bp | "
                       f"r_impl {t['Implied financing rate']:.3%} | IV c/p {t['Call implied vol']:.2%}/{t['Put implied vol']:.2%}")
else:
    _p("quote", "not filled")
print("Q4(c)")
for k, g in trk.groupby("cycle"):
    _p(f"Cycle {k}: gap syn-call at entry -> expiry (INR)", f"{g['gap_syn_call'].iloc[0]:.2f} -> {g['gap_syn_call'].iloc[-1]:.2f}; financed-call {g['gap_fin_call'].iloc[0]:.2f} -> {g['gap_fin_call'].iloc[-1]:.2f}")
_p("Cumulative P&L call / synthetic / financed", f"{trk['cum_pnl_call'].iloc[-1]:.2f} / {trk['cum_pnl_syn'].iloc[-1]:.2f} / {trk['cum_pnl_fin'].iloc[-1]:.2f}")
print("Supplement A"); _p("Ranking by QLIKE (best first)", ", ".join(VOL_STUDY["QLIKE"].sort_values().index))
_p("RMSE vol pts RV21 vs best", f"{VOL_STUDY.loc['RV21', 'RMSE (vol pts)']:.2f} vs {VOL_STUDY['RMSE (vol pts)'].min():.2f} ({VOL_STUDY['RMSE (vol pts)'].idxmin()})")
print("Supplement B"); _p("Spurious gap RMSE bp: q=0 / regular / incl. special", " / ".join(f"{DIV_STUDY.iloc[i]['Spurious gap, RMSE (bp)']:.1f}" for i in range(3)))
_p("Credit error mean INR: q=0 / regular / incl. special", " / ".join(f"{DIV_STUDY.iloc[i]['Credit error, mean (INR)']:.2f}" for i in range(3)))
print("Q5");  _p("Trades", f"{n_tr}")
for m in Q5_TABLE.index:
    _p(m, f"call {Q5_TABLE.loc[m, 'Actual Call']} | synthetic {Q5_TABLE.loc[m, 'Synthetic Call']} | winner {Q5_TABLE.loc[m, 'Winner']}")
_p("Synthetic - call mean (INR, bp of spot)", f"{(B - A).mean():.2f}, {((B - A) / trades['S0']).mean() * 1e4:.1f} bp")
for k, v in decomp.items():
    _p(f"  {k}", f"{v:.2f}")
_p("Financed - call: mean, sd, share beyond 25 bp", f"{diff.mean():.2f}, {diff.std(ddof=1):.2f}, {share_out:.1%}")
for q_ in Q5_TSTATS.index:
    t = Q5_TSTATS.loc[q_]
    _p(f"t-stats: {q_}", f"all {t['t, all trades']:.2f} | every 21st {t['t, every 21st (offset 0)']:.2f} "
                         f"(range {t['t, every 21st: min over 21 offsets']:.2f} to {t['t, every 21st: max over 21 offsets']:.2f}) | NW {t['Newey-West t (21 lags), all trades']:.2f}")
_p("Mean move, mean priced vol, mean realised vol over hold", f"{trades['move'].mean():.2%}, {trades['sigma_call'].mean():.2%}, {trades['rv_hold'].mean():.2%}")
for (st, cs), r_ in Q5C_TABLE.iterrows():
    _p(f"{st}, {cs}", f"{r_['Entry']} | move {r_['Stock move over hold']:.2%} | priced {r_['Priced-in vol at entry (call)']:.2%} | "
                      f"realised {r_['Realised vol over hold']:.2%} | P&L {r_['P&L (INR)']:.2f}")
_p("Q5 under q=0 / regular / special / causal: financed - call", " / ".join(f"{v:.2f}" for v in SENS_Q["Mean financed - call (INR)"]))
_p("Q5 under q=0 / regular / special / causal: mean call P&L", " / ".join(f"{v:.2f}" for v in SENS_Q["Mean call P&L (INR)"]))
_p("Distortion seeds: financed - call mean 5% / 50% / 95%", f"{SEEDS['mean'].quantile(.05):.2f} / {SEEDS['mean'].median():.2f} / {SEEDS['mean'].quantile(.95):.2f}")
_p("Distortion seeds: t every 21st, 5% / 50% / 95%", f"{SEEDS['t_21'].quantile(.05):.2f} / {SEEDS['t_21'].median():.2f} / {SEEDS['t_21'].quantile(.95):.2f}")
for lab, r_ in SPLIT.iterrows():
    _p(f"{lab} ({r_['From']} to {r_['To']})", f"call P&L {r_['Mean call P&L']:.2f} (t21 {r_['t call (every 21st)']:.2f}) | syn-call {r_['Mean syn - call']:.2f} | "
                                              f"fin-call {r_['Mean financed - call']:.2f} | move {r_['Mean move']:.2%} | vol priced {r_['Mean priced vol']:.2%} realised {r_['Mean realised vol']:.2%}")


**Your answer (Q5d) — at least 150 words:**

*Data provenance — underlying and option premiums:*

*The rupee advantage of the synthetic: size, meaning, and what financing does to it:*

*Return on capital, and which you would hold with ₹1 lakh and a bullish view:*

*What the best and worst trades have in common:*

*Independent observations, significance, and ignored frictions:*

---
## Q6 · Short answers · **[5 marks]**

Answer in **two to four sentences each**. Marks are for the mechanism, not the assertion.

1. Put-call parity is called **model-free**. What exactly does it assume, and what does it *not*
   assume that BSM does? If you observe a parity violation in single-stock options, does that tell
   you that BSM is wrong?

2. You priced every option in Q5 from **historical** volatility. In real markets, implied
   volatility is on average higher than the volatility subsequently realised. What does that do
   to the average P&L of a strategy that systematically **buys** options, and which of your Q5
   results would change in sign or size?

3. A classmate says: *"The binomial model is just a crude approximation to Black-Scholes, so
   there is no reason ever to use it."* Give two specific situations in which the tree is the
   better tool, and say what the tree can do in each that the formula cannot.

4. The synthetic call needs you to **own the stock**; the mirror-image synthetic **put**
   (long call + short stock) needs you to **short** it. Explain why, in practice, parity violations
   are larger and more persistent in one direction than the other, and which direction that is.

| Each sub-question | Marks |
|---|---|
| Correct conclusion **and** the mechanism behind it, within the length limit | 2 |
| Correct conclusion, mechanism vague or missing; or right mechanism with a wrong sign | 1 |
| Assertion only, or more than twice the length limit without a clear answer | 0 – 0.5 |

**Your answers (Q6):**

1. Put-call parity assumes only that the call and the put are European with the same strike and expiry, that the stock can be bought or sold short and money borrowed or lent at $r$, that dividends over the option's life are known (here through $q$), and that there are no frictions. It assumes nothing about how the price moves: no lognormal distribution, no constant volatility and no continuous hedging, all of which BSM needs. NSE single-stock options are European, so a parity violation does not show that BSM is wrong. It shows that one of the parity conditions fails in practice, usually a short-sale limit, a financing rate different from $r$, a dividend estimate, the bid-ask spread or quotes taken at different moments.

2. When implied volatility is above the volatility later realised, an option buyer pays for more variance than the stock delivers, so a strategy that keeps buying options loses that premium on average; the loss per trade is roughly vega times the gap in volatility points. My Q5 premiums were set at historical volatility, so this premium is missing. Repriced at implied volatility, the mean P&L and ROC of the actual call would fall and could turn negative, and the synthetic's put leg would cost more by about the same vega amount, lowering its P&L as well. The financed-minus-call difference would hardly move, because parity ties the two premiums together; it changes only by the gap between the call and put implied volatilities.

3. First, American options: the tree compares exercising now with holding at every node, which no closed form does for an American put, so it prices early exercise and shows where exercise is optimal. Second, discrete dividends of known size and date, such as TCS's ₹46 special dividend of 16 Jan 2026: the tree can drop the stock by the cash amount at the ex-date node, while BSM can only spread the dividend as a continuous yield $q$, which misprices a short-dated option whose life does or does not include the ex-date.

4. The synthetic call (long stock and long put), and the conversion that sells an expensive call against it, need only a stock purchase, so calls that are too expensive relative to puts are arbitraged away quickly. The opposite trade, a reversal that sells the expensive put, buys the call and shorts the stock, needs an overnight short position. In India a retail short sale in the cash market must be covered the same day, and an overnight short needs stock borrowed through SLB, which is thin and costly. So violations in which puts are expensive relative to calls ($C - P$ below $Se^{-qT} - Ke^{-rT}$, or an implied financing rate below $r$) are larger and last longer.


---
## Declaration and checklist

**Sources, collaborators and any AI assistance used** (be specific — which tool, for which part,
and what you did with the output). Full Disclosure of the prompt is expected and costs you nothing; undisclosed use
of AI will be considered as an integrity violation.

*Your declaration:*

---

### Before you submit, confirm every box

- [ ] `NAME`, `CAMPUS_ID`, `TICKER` and `HONOUR` are filled in and the identification cell runs
- [ ] The analysis is on the stock **allotted to you**
- [ ] `Kernel → Restart & Run All` completes with **no errors**
- [ ] Every plot and table is **visible in the saved file** (not an empty output cell)
- [ ] Every `**Your answer:**` cell is filled in — none left blank
- [ ] You stated, for each data series, whether it is **live or simulated**, and for each option
      price whether it is **observed or model-generated**
- [ ] All assertions in Q2 and Q3 are present and pass
- [ ] Sources cited for: dividend yield; the observed call and put quote (with timestamp)
- [ ] Q5(d) is at least 150 words
- [ ] Saved as `DRM_Lab04_<CampusID>.ipynb`
- [ ] Uploaded to the Lab 04 assignment on **Nalanda** before the deadline

**DRAFT. Edit every line so that it states only what you actually did. Delete this line and the square brackets when done.**

**AI assistance.** Tool: Claude (Anthropic), used in one Claude session with access to my repository `Tarush-035/DRM_Sem1_26_27`
(branch `lab04-assignment03`) on 24-Nov-2026. Model Claude Sonnet 5.5 for the first build of the code (Phase 1); model Claude Opus 5.5 for
the revision that added the NSE dividend file, the Q4(b) quote, the Q6 drafts and the written answers.
[If you used any other AI tool, for example to read the NSE option chain or the corporate-action filters, name it here and say for what.]

**Parts where it was used.**
- Code for Q1 to Q5 and for three supplements (A: volatility-estimator comparison, B: treatment of the special dividends in $q$, C: Q5 sensitivity tables).
- The corporate-action and dividend-calendar research for TCS. I then downloaded the NSE corporate-actions CSV myself and the dividend table now follows it.
- A first draft of the written answers: Q6 (conceptual, written before my live run) and Q1c, Q2, Q3d, Q4d, Q5d (written after I pasted my own live outputs).
- This declaration draft.

**My own earlier code reused.** Functions from my `edhec_risk_kit.py` (written in an earlier course): `drawdown`, `skewness`, `kurtosis`, `is_normal`.
They were adapted as listed in the comment at the top of the Q1 code cell.

**What I did with the output.**
- [I read every code cell and can explain it line by line. Confirm or edit.]
- [I ran `Restart & Run All` on my own machine on DATE with live data from Yahoo Finance and FRED. Confirm or edit.]
- [I fetched the Q4(b) call and put quote myself from the NSE option chain (TCS, strike 2,460, expiry 29-Dec-2026, 23-Nov-2026 15:40 IST) and saved the CSV and a screenshot. Confirm or edit.]
- [I downloaded the NSE corporate-actions CSV for TCS on 24-Nov-2026; the dividend table in the notebook matches it. Confirm or edit.]
- [I rewrote or changed these parts of the written answers: LIST. Confirm or edit.]

**Data and premiums.** Prices: Yahoo Finance via `yfinance`, unadjusted closes. Rate: FRED series `INDIRLTLT01STM`. Option premiums in Q2 to Q5 are model-generated;
the Q4(b) premiums are observed. Where a run fell back to simulated data the outputs are labelled `[SIM]` and are not market data.

**Prompt given to the AI assistant (included in full).**

<details><summary>Click to expand the full prompt</summary>

```text
ROLE
You are a senior quantitative finance developer helping me complete a graded
assignment: BITS Pilani, ECON F354 Derivatives and Risk Management, Lab 04
Assignment 03 (Option Pricing and Synthetics). The notebook is
Lab04_Assignment_03_Option_Pricing_and_Synthetics.ipynb in my repo
Tarush-035/DRM_Sem1_26_27. Work on a new branch named lab04-assignment03.
Never commit to main.

MY INPUTS
- Ticker: <TCS>. Analyse only this stock. Also, give me a brief explanation as to why this stock is a good choice, I have picked it because there is no special corporate action in the 2 year window the question requires us to work on.
- I will fill NAME, CAMPUS_ID, EMAIL and HONOUR myself. Leave them blank.
- My earlier code is pasted at the bottom. Reuse it where it fits, adapt it
  to this notebook's conventions, and list what you changed.

READ FIRST
Read the whole notebook, including the rubric at the top and the criterion
table under every question. The rubric is the spec. Keep the provided
data-loading cells exactly as given. Do not delete question cells.

RULES FROM THE SHEET
1. Closes are unadjusted. Before any analysis, flag every day with an absolute
   daily log return above 8% and name the corporate action behind it. If a
   split, bonus or demerger sits in the window, stop and tell me.
2. No look-ahead: volatility on date t uses returns up to t only.
3. Expiry = last trading day on or before entry + 30 calendar days (searchsorted
   on the price index). Drop trades whose expiry is beyond the data and report
   the remaining count.
4. Credit the stock leg of the synthetic with dividends S0*(exp(qT)-1), using the
   same q as Q2. q needs a named, dated source.
5. Label every series [LIVE] or [SIM] and every option price observed or
   model-generated. Never present simulated numbers as market data.
6. Q2: three passing assertions (textbook benchmark, parity to 1e-8 with q != 0,
   finite-difference check of delta and vega).
7. Q3: CRR tree, vectorised per time slice, validity check on p, parity
   assertion at N=50, N from 1 to 400, and a demonstrated fix for the
   odd-even oscillation.
8. Q4(b): leave the observed-quote constants for me to fill from the NSE option
   chain. The code must run once I fill them. Never invent a quote.
9. Q5: premiums are model-generated unless I supply real ones. Name every
   parameter of the put-volatility distortion as a constant. Report t-statistics
   on all trades and on every 21st trade.
10. Every chart has a title, axis labels with units and a legend.
11. Restart & Run All must finish with no errors and saved outputs. Test the
    full notebook in your sandbox on the simulated fallback, and tell me which
    outputs came from it.

TWO PHASES
Phase 1: build and test all code, push to the branch, and give me a short list
of what to run locally and what to fetch from NSE.
Phase 2: after I paste the live outputs, write every answer cell. Each answer
names the mechanism, quotes my own numbers and states data provenance. Do not
write numbers into answers before Phase 2.

DISCLOSURE AND VIVA
Comment every code block so I can explain it line by line. Draft the final
declaration cell: which tool, which parts, what I reviewed, and that this
prompt is included in full.

STYLE
Plain professional prose. No em dashes. No padding words such as leverage,
robust, nuanced or seamless. Concrete numbers over abstract summaries.

MY EARLIER CODE - I have pasted the EDHEC risk kit, it might have code that I wrote in machine learnign courses that will be directly used here, so you don't ahve to write everythign on your own.

In the latest drm project, how about rather than using one estimate of volatility from 2 year average, 21 day rollign window or so on, we use a weighted average of time delay, like or something like an array of different volatility and see somewhere, which one predicted the best among these oprions too? or we can first generate a first notebook adn then see which estimate is best, adn then implement it with the other estimates, fully disclosing that we tried all estimates.
 Also once we decide on how to use the special divident, we willl have to stick with it for Q5 too, before jumping on to any conclusion, qualitatively tell me which one would be better, or if you want you can run simulations of your own to help tell me which one of these would be better.

```

Follow-up messages in the same session (summarised, not instructions to the grader): I uploaded my Lab 03 submission, the NSE option-chain CSVs
for TCS (expiries 23-Nov-2026 and 29-Dec-2026), the NSE corporate-actions CSVs and a screenshot of the option chain, and asked for the complete
notebook to download and run, and which model was used. Later messages pasted my live outputs and asked for the written answers.

This is the final version of the first prompt. An earlier message with the same text and the placeholder `<TICKER>` was interrupted before it was answered.
The file `edhec_risk_kit.py` that I attached to it is my own earlier code and is not reproduced here.

</details>
